# Last-Mile Route Optimization

We compare two OR-Tools heuristics for one-vehicle delivery routes:

1. **Travel objective:** minimize estimated driving time.
2. **Elapsed objective:** minimize driving + service + waiting time.

Both versions start at the station, visit every assigned stop once, respect package time windows, use direction-specific travel times, and do not return to the station. Service durations are summed across packages assigned to the same stop.

The notebook uses the first 20 dataset routes for input validation and a five-route subset for the default optimization comparison. This Amazon dataset is US-based; results are a method demonstration and should not be presented as measured Indian operating savings.


In [ ]:
import json
from pathlib import Path

relative_data_path = Path("data/raw/amazon/almrrc2021-data-training/model_build_inputs")
if not relative_data_path.exists():
    relative_data_path = Path("../data/raw/amazon/almrrc2021-data-training/model_build_inputs")

data_dir = relative_data_path.resolve()
if not data_dir.exists():
    raise FileNotFoundError(f"Amazon training data folder not found: {data_dir}")

pilot_route_id = "RouteID_00575ca4-8a63-49d2-96c8-9b347be5ba6c"

with (data_dir / "route_data.json").open(encoding="utf-8") as file:
    route_json = json.load(file)

with (data_dir / "actual_sequences.json").open(encoding="utf-8") as file:
    actual_sequences = json.load(file)

evaluation_route_ids = list(route_json)[:20]

pilot_route = route_json[pilot_route_id]
pilot_sequence = actual_sequences[pilot_route_id]["actual"]
ordered_stop_ids = [
    stop_id
    for stop_id, position in sorted(pilot_sequence.items(), key=lambda item: item[1])
]
route_stop_ids = set(pilot_route["stops"])
dropoff_ids = {
    stop_id for stop_id, stop in pilot_route["stops"].items()
    if stop["type"] == "Dropoff"
}

print("Pilot route:", pilot_route_id)
print("Station:", pilot_route["station_code"])
print("Total route stops:", len(route_stop_ids))
print("Delivery stops:", len(dropoff_ids))
print("Sequence covers all stops:", set(ordered_stop_ids) == route_stop_ids)
print("Sequence starts at station:", pilot_route["stops"][ordered_stop_ids[0]]["type"] == "Station")


In [ ]:
import ijson
import re

package_file = data_dir / "package_data.json"
travel_file = data_dir / "travel_times.json"


class NaNReplacingReader:
    """Replace Amazon's non-standard NaN tokens while streaming bytes to ijson."""
    def __init__(self, raw_file, block_size=1_000_000):
        self.raw_file = raw_file
        self.block_size = block_size
        self.pending = b""
        self.ready = bytearray()
        self.eof = False

    def read(self, size=-1):
        if size == 0:
            return b""
        if size < 0:
            raise ValueError("The streaming parser requested an unbounded read.")

        while len(self.ready) < size and not self.eof:
            chunk = self.raw_file.read(max(self.block_size, size - len(self.ready)))
            if not chunk:
                self.ready.extend(re.sub(rb"\bNaN\b", b"null", self.pending))
                self.pending = b""
                self.eof = True
                break

            combined = self.pending + chunk
            safe_length = max(0, len(combined) - 3)
            safe_part = combined[:safe_length]
            self.pending = combined[safe_length:]
            self.ready.extend(re.sub(rb"\bNaN\b", b"null", safe_part))

        output = bytes(self.ready[:size])
        del self.ready[:size]
        return output


route_id_set = set(evaluation_route_ids)
evaluation_packages = {}
evaluation_matrices = {}

# ijson parses quickly and keeps memory use low. Each large file is read once.
with package_file.open("rb") as raw_file:
    for route_id, packages_by_stop in ijson.kvitems(
        NaNReplacingReader(raw_file), "", use_float=True
    ):
        if route_id in route_id_set:
            evaluation_packages[route_id] = packages_by_stop
        if len(evaluation_packages) == len(route_id_set):
            break

with travel_file.open("rb") as raw_file:
    for route_id, matrix in ijson.kvitems(
        NaNReplacingReader(raw_file), "", use_float=True
    ):
        if route_id in route_id_set:
            evaluation_matrices[route_id] = matrix
        if len(evaluation_matrices) == len(route_id_set):
            break

missing_packages = route_id_set - set(evaluation_packages)
missing_matrices = route_id_set - set(evaluation_matrices)
if missing_packages or missing_matrices:
    raise ValueError(
        f"Missing selected routes: packages={len(missing_packages)}, "
        f"travel matrices={len(missing_matrices)}"
    )

pilot_packages_by_stop = evaluation_packages[pilot_route_id]
pilot_travel_matrix = evaluation_matrices[pilot_route_id]

print("Selected routes loaded:", len(route_id_set))
print("Package records loaded:", len(evaluation_packages))
print("Travel matrices loaded:", len(evaluation_matrices))
print("Pilot package stops match route:", set(pilot_packages_by_stop) == route_stop_ids)
print("Pilot matrix stops match route:", set(pilot_travel_matrix) == route_stop_ids)


In [ ]:
# Historical sequence mein consecutive stops ke beech ka travel time jodo
baseline_travel_seconds = sum(
    pilot_travel_matrix[origin][destination]
    for origin, destination in zip(
        ordered_stop_ids,
        ordered_stop_ids[1:]
    )
)

# Sirf delivery stops ke packages ka planned service time jodo
baseline_service_seconds = sum(
    details["planned_service_time_seconds"]
    for stop_id in ordered_stop_ids
    if pilot_route["stops"][stop_id]["type"] == "Dropoff"
    for details in pilot_packages_by_stop[stop_id].values()
)

print("Historical stops in sequence:", len(ordered_stop_ids))
print("Starts at station:",
      pilot_route["stops"][ordered_stop_ids[0]]["type"] == "Station")
print("Estimated travel time (minutes):",
      round(baseline_travel_seconds / 60, 1))
print("Planned delivery service time (minutes):",
      round(baseline_service_seconds / 60, 1))
print(
    "Estimated travel + service time (minutes):",
    round((baseline_travel_seconds + baseline_service_seconds) / 60, 1)
)

In [ ]:
from datetime import datetime, timedelta, timezone

def parse_utc(value):
    # ISO time ko UTC datetime mein badlo
    value = value.replace("Z", "+00:00")
    parsed = datetime.fromisoformat(value)

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    return parsed.astimezone(timezone.utc)


departure_time = datetime.fromisoformat(
    f"{pilot_route['date_YYYY_MM_DD']}T"
    f"{pilot_route['departure_time_utc']}+00:00"
)

service_seconds_by_stop = {}
window_by_stop = {}

for stop_id in ordered_stop_ids:
    if pilot_route["stops"][stop_id]["type"] != "Dropoff":
        continue

    packages = pilot_packages_by_stop[stop_id]

    # Stop par saare packages ka planned service time jodo
    service_seconds_by_stop[stop_id] = sum(
        details["planned_service_time_seconds"]
        for details in packages.values()
    )

    # Stop ke windowed packages ka common time window nikalo
    package_windows = []

    for details in packages.values():
        time_window = details.get("time_window") or {}
        start = time_window.get("start_time_utc")
        end = time_window.get("end_time_utc")

        if isinstance(start, str) and isinstance(end, str):
            package_windows.append((
                parse_utc(start),
                parse_utc(end)
            ))

    if package_windows:
        shared_start = max(start for start, end in package_windows)
        shared_end = min(end for start, end in package_windows)

        if shared_start > shared_end:
            raise ValueError(
                f"Packages at stop {stop_id} have no shared time window."
            )

        window_by_stop[stop_id] = (shared_start, shared_end)


current_time = departure_time
total_wait_seconds = 0
late_stops = []

for origin, destination in zip(
    ordered_stop_ids,
    ordered_stop_ids[1:]
):
    travel_seconds = pilot_travel_matrix[origin][destination]
    arrival_time = current_time + timedelta(seconds=travel_seconds)
    service_start_time = arrival_time

    if destination in window_by_stop:
        window_start, window_end = window_by_stop[destination]

        if service_start_time < window_start:
            wait_seconds = (window_start - service_start_time).total_seconds()
            total_wait_seconds += wait_seconds
            service_start_time = window_start

        if service_start_time > window_end:
            late_stops.append(destination)

    current_time = service_start_time + timedelta(
        seconds=service_seconds_by_stop[destination]
    )


print("Departure UTC:", departure_time.isoformat())
print("Delivery stops with time windows:", len(window_by_stop))
print("Total waiting time (minutes):", round(total_wait_seconds / 60, 1))
print("Stops where service starts after window ends:", len(late_stops))
print("Estimated route finish UTC:", current_time.isoformat())
print(
    "Estimated elapsed time, including waiting (minutes):",
    round((current_time - departure_time).total_seconds() / 60, 1)
)

In [ ]:
import pandas as pd

# Historical order mein station aur delivery stops rakho
optimization_stop_ids = ordered_stop_ids.copy()

optimizer_rows = []

for stop_id in optimization_stop_ids:
    stop_type = pilot_route["stops"][stop_id]["type"]

    if stop_type == "Station":
        service_seconds = 0
    else:
        service_seconds = service_seconds_by_stop[stop_id]

    time_window = window_by_stop.get(stop_id)

    if time_window is None:
        window_start_seconds = None
        window_end_seconds = None
    else:
        window_start, window_end = time_window

        # Window times ko departure se guzre seconds mein badlo
        window_start_seconds = max(
            0,
            int((window_start - departure_time).total_seconds())
        )
        window_end_seconds = int(
            (window_end - departure_time).total_seconds()
        )

    optimizer_rows.append({
        "stop_id": stop_id,
        "stop_type": stop_type,
        "service_seconds": service_seconds,
        "has_time_window": time_window is not None,
        "window_start_seconds": window_start_seconds,
        "window_end_seconds": window_end_seconds,
    })

optimizer_stops_df = pd.DataFrame(optimizer_rows)

# Travel matrix ki rows aur columns ko bhi isi stop order mein rakho
travel_time_matrix_seconds = [
    [
        pilot_travel_matrix[origin][destination]
        for destination in optimization_stop_ids
    ]
    for origin in optimization_stop_ids
]

print("Stops prepared:", len(optimization_stop_ids))
print("Delivery stops:", sum(
    stop_type == "Dropoff"
    for stop_type in optimizer_stops_df["stop_type"]
))
print("Stops with time windows:",
      optimizer_stops_df["has_time_window"].sum())
print(
    "Travel matrix dimensions:",
    len(travel_time_matrix_seconds),
    "rows x",
    len(travel_time_matrix_seconds[0]),
    "columns"
)

display(optimizer_stops_df.head(10))

In [ ]:
import importlib.util

print(importlib.util.find_spec("ortools"))

In [ ]:
import ortools
print("OR-Tools is ready:", ortools.__version__)

In [ ]:
# Confirm the selected route, package, matrix, and sequence records align.
input_checks = []

for route_id in evaluation_route_ids:
    expected_stop_ids = set(route_json[route_id]["stops"])
    checks = {
        "route_id": route_id,
        "stop_count": len(expected_stop_ids),
        "package_stops_match": set(evaluation_packages[route_id]) == expected_stop_ids,
        "matrix_origins_match": set(evaluation_matrices[route_id]) == expected_stop_ids,
        "matrix_rows_match": all(
            set(row) == expected_stop_ids
            for row in evaluation_matrices[route_id].values()
        ),
        "sequence_matches_stops": set(actual_sequences[route_id]["actual"]) == expected_stop_ids,
    }
    input_checks.append(checks)

evaluation_checks_df = pd.DataFrame(input_checks)
print("Routes selected:", len(evaluation_route_ids))
print("All input checks passed:", bool(evaluation_checks_df.drop(columns="route_id").select_dtypes(include="bool").all().all()))
display(evaluation_checks_df)


In [ ]:
from datetime import datetime, timezone

from ortools.constraint_solver import pywrapcp, routing_enums_pb2


def optimize_one_route(
    route_id,
    time_limit_seconds=15,
    objective="travel",
):
    """
    Optimize one route.

    objective="travel":
        Minimize estimated driving time.

    objective="elapsed":
        Minimize elapsed time first, with driving time as a tie-breaker.
    """

    if time_limit_seconds <= 0:
        raise ValueError("time_limit_seconds must be greater than zero.")

    if objective not in {"travel", "elapsed"}:
        raise ValueError("objective must be 'travel' or 'elapsed'.")

    scale = 10  # Use integer deciseconds inside OR-Tools.

    route = route_json[route_id]
    packages_by_stop = evaluation_packages[route_id]
    travel_matrix = evaluation_matrices[route_id]

    sequence = actual_sequences[route_id]["actual"]
    ordered_stop_ids = [
        stop_id
        for stop_id, position in sorted(
            sequence.items(),
            key=lambda item: item[1],
        )
    ]

    expected_stop_ids = set(route["stops"])

    if not ordered_stop_ids:
        raise ValueError(f"Route {route_id} has no stops.")

    if set(ordered_stop_ids) != expected_stop_ids:
        raise ValueError(f"Sequence does not match stops for {route_id}.")

    station_id = ordered_stop_ids[0]

    if route["stops"][station_id]["type"] != "Station":
        raise ValueError(
            f"Historical sequence does not start at the station for {route_id}."
        )

    if set(packages_by_stop) != expected_stop_ids:
        raise ValueError(f"Package stops do not match for {route_id}.")

    if set(travel_matrix) != expected_stop_ids:
        raise ValueError(f"Travel-matrix origins do not match for {route_id}.")

    for origin in expected_stop_ids:
        if set(travel_matrix[origin]) != expected_stop_ids:
            raise ValueError(
                f"Travel-matrix destinations do not match at stop {origin}."
            )

    departure_time = datetime.fromisoformat(
        f"{route['date_YYYY_MM_DD']}T"
        f"{route['departure_time_utc']}+00:00"
    )

    def parse_utc(value):
        value = value.replace("Z", "+00:00")
        parsed = datetime.fromisoformat(value)

        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)

        return parsed.astimezone(timezone.utc)

    # Combine package service durations and package windows by stop.
    service_seconds_by_stop = {station_id: 0.0}
    window_seconds_by_stop = {}

    for stop_id in ordered_stop_ids[1:]:
        if route["stops"][stop_id]["type"] != "Dropoff":
            raise ValueError(f"Unexpected non-delivery stop: {stop_id}")

        service_times = []
        package_windows = []

        for details in packages_by_stop[stop_id].values():
            service_time = details.get("planned_service_time_seconds")

            if service_time is None:
                raise ValueError(f"Missing service time at stop {stop_id}.")

            service_times.append(float(service_time))

            time_window = details.get("time_window") or {}
            start = time_window.get("start_time_utc")
            end = time_window.get("end_time_utc")

            has_start = isinstance(start, str)
            has_end = isinstance(end, str)

            if has_start != has_end:
                raise ValueError(f"Partial time window at stop {stop_id}.")

            if has_start and has_end:
                start_time = parse_utc(start)
                end_time = parse_utc(end)

                if end_time < start_time:
                    raise ValueError(f"Invalid time window at stop {stop_id}.")

                package_windows.append((start_time, end_time))

        service_seconds_by_stop[stop_id] = sum(service_times)

        if package_windows:
            shared_start = max(start for start, end in package_windows)
            shared_end = min(end for start, end in package_windows)

            if shared_start > shared_end:
                raise ValueError(
                    f"Packages at stop {stop_id} have no shared time window."
                )

            window_seconds_by_stop[stop_id] = (
                (shared_start - departure_time).total_seconds(),
                (shared_end - departure_time).total_seconds(),
            )

    number_of_stops = len(ordered_stop_ids)
    dummy_end_node = number_of_stops

    travel_times_int = [
        [
            int(round(travel_matrix[origin][destination] * scale))
            for destination in ordered_stop_ids
        ]
        for origin in ordered_stop_ids
    ]

    service_times_int = [
        int(round(service_seconds_by_stop[stop_id] * scale))
        for stop_id in ordered_stop_ids
    ]

    time_windows_int = []

    for stop_id in ordered_stop_ids:
        if stop_id not in window_seconds_by_stop:
            time_windows_int.append(None)
            continue

        start_seconds, end_seconds = window_seconds_by_stop[stop_id]
        lower_bound = int(round(max(0, start_seconds) * scale))
        upper_bound = int(round(end_seconds * scale))

        if lower_bound > upper_bound:
            raise ValueError(
                f"Time window ends before departure at stop {stop_id}."
            )

        time_windows_int.append((lower_bound, upper_bound))

    # Add a dummy end so the vehicle does not return to the station.
    for row in travel_times_int:
        row.append(0)

    travel_times_int.append([0] * (number_of_stops + 1))
    service_times_int.append(0)
    time_windows_int.append(None)

    max_window_end = max(
        (
            bounds[1]
            for bounds in time_windows_int[:number_of_stops]
            if bounds is not None
        ),
        default=0,
    )
    max_travel_time = max(max(row) for row in travel_times_int)
    total_service_time = sum(service_times_int)

    horizon = max(
        24 * 60 * 60 * scale,
        max_window_end
        + total_service_time
        + (number_of_stops + 1) * max_travel_time,
    )

    manager = pywrapcp.RoutingIndexManager(
        number_of_stops + 1,
        1,
        [0],
        [dummy_end_node],
    )
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        if to_node == dummy_end_node:
            return service_times_int[from_node]

        return (
            service_times_int[from_node]
            + travel_times_int[from_node][to_node]
        )

    time_callback_index = routing.RegisterTransitCallback(time_callback)

    routing.AddDimension(
        time_callback_index,
        horizon,
        horizon,
        True,
        "Time",
    )

    time_dimension = routing.GetDimensionOrDie("Time")

    def travel_cost_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        if to_node == dummy_end_node:
            return 0

        return travel_times_int[from_node][to_node]

    travel_cost_index = routing.RegisterTransitCallback(
        travel_cost_callback
    )
    routing.SetArcCostEvaluatorOfAllVehicles(travel_cost_index)

    if objective == "elapsed":
        # Elapsed time is the primary goal; driving time breaks ties.
        time_dimension.SetGlobalSpanCostCoefficient(horizon + 1)

    time_dimension.CumulVar(routing.Start(0)).SetRange(0, 0)

    for node in range(1, number_of_stops):
        index = manager.NodeToIndex(node)
        bounds = time_windows_int[node]

        if bounds is None:
            time_dimension.CumulVar(index).SetRange(0, horizon)
        else:
            time_dimension.CumulVar(index).SetRange(*bounds)

    time_dimension.CumulVar(routing.End(0)).SetRange(0, horizon)
    routing.AddVariableMinimizedByFinalizer(
        time_dimension.CumulVar(routing.End(0))
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()

    # Change from the previous version:
    # use the VRPTW example's PATH_CHEAPEST_ARC first-route strategy.
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = int(time_limit_seconds)

    solution = routing.SolveWithParameters(search_parameters)
    solver_status = routing.status()

    solver_status_names = {
        0: "not solved",
        1: "success",
        2: "partial success",
        3: "failure",
        4: "time limit reached before solution",
        5: "invalid model or parameters",
        6: "proven infeasible",
    }
    solver_status_name = solver_status_names.get(
        solver_status,
        f"unknown status {solver_status}",
    )

    if solution is None:
        return {
            "route_id": route_id,
            "objective": objective,
            "status": solver_status_name,
            "solver_status": solver_status,
        }

    optimized_stop_ids = []
    optimized_window_violations = []
    index = routing.Start(0)

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        stop_id = ordered_stop_ids[node]
        optimized_stop_ids.append(stop_id)

        bounds = time_windows_int[node]

        if bounds is not None:
            arrival = solution.Value(time_dimension.CumulVar(index))

            if not bounds[0] <= arrival <= bounds[1]:
                optimized_window_violations.append(stop_id)

        index = solution.Value(routing.NextVar(index))

    historical_travel_seconds = sum(
        travel_matrix[origin][destination]
        for origin, destination in zip(
            ordered_stop_ids,
            ordered_stop_ids[1:],
        )
    )

    optimized_travel_seconds = sum(
        travel_matrix[origin][destination]
        for origin, destination in zip(
            optimized_stop_ids,
            optimized_stop_ids[1:],
        )
    )

    historical_service_seconds = sum(service_seconds_by_stop.values())

    current_time = 0.0
    historical_wait_seconds = 0.0
    historical_late_stops = []

    for origin, destination in zip(
        ordered_stop_ids,
        ordered_stop_ids[1:],
    ):
        arrival = current_time + travel_matrix[origin][destination]
        service_start = arrival

        if destination in window_seconds_by_stop:
            start_seconds, end_seconds = window_seconds_by_stop[destination]
            service_start = max(arrival, start_seconds)
            historical_wait_seconds += service_start - arrival

            if service_start > end_seconds:
                historical_late_stops.append(destination)

        current_time = (
            service_start + service_seconds_by_stop[destination]
        )

    optimized_elapsed_seconds = (
        solution.Value(time_dimension.CumulVar(routing.End(0))) / scale
    )

    optimized_wait_seconds = max(
        0.0,
        optimized_elapsed_seconds
        - optimized_travel_seconds
        - historical_service_seconds,
    )

    return {
        "route_id": route_id,
        "objective": objective,
        "objective_definition": (
            "elapsed time primary, driving time tie-breaker"
            if objective == "elapsed"
            else "minimize driving time"
        ),
        "status": "feasible solution found",
        "solver_status": solver_status,
        "stop_count": number_of_stops,
        "historical_travel_minutes": historical_travel_seconds / 60,
        "optimized_travel_minutes": optimized_travel_seconds / 60,
        "travel_reduction_minutes": (
            historical_travel_seconds - optimized_travel_seconds
        ) / 60,
        "travel_reduction_percent": (
            (historical_travel_seconds - optimized_travel_seconds)
            / historical_travel_seconds
            * 100
        ),
        "historical_elapsed_minutes": current_time / 60,
        "historical_wait_minutes": historical_wait_seconds / 60,
        "optimized_elapsed_minutes": optimized_elapsed_seconds / 60,
        "optimized_wait_minutes": optimized_wait_seconds / 60,
        "historical_late_stops": len(historical_late_stops),
        "optimized_window_violations": len(
            optimized_window_violations
        ),
        "all_stops_visited_once": (
            len(optimized_stop_ids) == number_of_stops
            and set(optimized_stop_ids) == expected_stop_ids
        ),
        "optimized_stop_ids": optimized_stop_ids,
    }

In [ ]:
from datetime import datetime, timezone

def parse_utc_for_audit(value):
    value = value.replace("Z", "+00:00")
    parsed = datetime.fromisoformat(value)

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    return parsed.astimezone(timezone.utc)


historical_audit_rows = []

for route_id in evaluation_route_ids:
    route = route_json[route_id]
    packages_by_stop = evaluation_packages[route_id]
    travel_matrix = evaluation_matrices[route_id]

    sequence = actual_sequences[route_id]["actual"]
    historical_stop_ids = [
        stop_id
        for stop_id, position in sorted(
            sequence.items(),
            key=lambda item: item[1]
        )
    ]

    departure_time = datetime.fromisoformat(
        f"{route['date_YYYY_MM_DD']}T"
        f"{route['departure_time_utc']}+00:00"
    )

    service_seconds_by_stop = {}
    window_seconds_by_stop = {}

    # Har delivery stop ke package service times aur shared window jodo
    for stop_id in historical_stop_ids:
        if route["stops"][stop_id]["type"] != "Dropoff":
            service_seconds_by_stop[stop_id] = 0.0
            continue

        packages = packages_by_stop[stop_id]

        service_seconds_by_stop[stop_id] = sum(
            details["planned_service_time_seconds"]
            for details in packages.values()
        )

        package_windows = []

        for details in packages.values():
            time_window = details.get("time_window") or {}
            start = time_window.get("start_time_utc")
            end = time_window.get("end_time_utc")

            if isinstance(start, str) and isinstance(end, str):
                package_windows.append((
                    parse_utc_for_audit(start),
                    parse_utc_for_audit(end)
                ))

        if package_windows:
            shared_start = max(start for start, end in package_windows)
            shared_end = min(end for start, end in package_windows)

            if shared_start > shared_end:
                raise ValueError(
                    f"No shared window for {route_id}, stop {stop_id}"
                )

            window_seconds_by_stop[stop_id] = (
                (shared_start - departure_time).total_seconds(),
                (shared_end - departure_time).total_seconds(),
            )

    # Historical sequence ka driving time
    historical_travel_seconds = sum(
        travel_matrix[origin][destination]
        for origin, destination in zip(
            historical_stop_ids,
            historical_stop_ids[1:]
        )
    )

    # Historical order ko departure se simulate karo
    current_time = 0.0
    waiting_seconds = 0.0
    late_stops = []

    for origin, destination in zip(
        historical_stop_ids,
        historical_stop_ids[1:]
    ):
        arrival_time = (
            current_time + travel_matrix[origin][destination]
        )
        service_start_time = arrival_time

        if destination in window_seconds_by_stop:
            window_start, window_end = window_seconds_by_stop[destination]
            service_start_time = max(arrival_time, window_start)
            waiting_seconds += service_start_time - arrival_time

            if service_start_time > window_end:
                late_stops.append(destination)

        current_time = (
            service_start_time + service_seconds_by_stop[destination]
        )

    historical_audit_rows.append({
        "route_id": route_id,
        "stop_count": len(historical_stop_ids),
        "windowed_stops": len(window_seconds_by_stop),
        "historical_travel_minutes": historical_travel_seconds / 60,
        "historical_wait_minutes": waiting_seconds / 60,
        "historical_elapsed_minutes": current_time / 60,
        "historical_late_stops": len(late_stops),
        "historical_time_windows_feasible": len(late_stops) == 0,
    })


historical_audit_df = pd.DataFrame(historical_audit_rows)

print("Routes audited:", len(historical_audit_df))
print(
    "Historical routes with no late stops:",
    historical_audit_df["historical_time_windows_feasible"].sum()
)
print(
    "Historical routes with late stops:",
    (~historical_audit_df["historical_time_windows_feasible"]).sum()
)

display(historical_audit_df)

## Short objective comparison

This is the default experiment: five routes, each with a 5-second solver limit. It compares a travel-time solution with an elapsed-time solution using the same solver start strategy. The 20-route, 15-second-per-route experiment is intentionally removed from the main workflow because it takes several minutes.

These are heuristic solutions, not guaranteed global optima. The first 20 routes are a convenience sample from the dataset, not a representative sample of Indian delivery operations.


In [ ]:
quick_route_ids = [
    pilot_route_id,
    "RouteID_00573a3d-b909-42ee-8a99-c6474309d96b",  # high historical waiting
    "RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c",
    "RouteID_0091a35c-85fa-4b5e-90cf-4778f441753e",
    "RouteID_00a25e47-65c9-4efc-bfe9-8d9648a0df4e",
]
quick_route_ids = [route_id for route_id in quick_route_ids if route_id in evaluation_route_ids]

quick_travel_results = []
for number, route_id in enumerate(quick_route_ids, start=1):
    print(f"Travel objective {number}/{len(quick_route_ids)}: {route_id}")
    result = optimize_one_route(
        route_id,
        time_limit_seconds=5,
        objective="travel",
    )
    quick_travel_results.append(result)
    print("Status:", result["status"])

quick_travel_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "optimized_stop_ids"}
    for result in quick_travel_results
])

quick_feasible_travel = quick_travel_df[
    quick_travel_df["status"] == "feasible solution found"
]
print("Routes tested:", len(quick_travel_df))
print("Feasible travel solutions:", len(quick_feasible_travel))
print("Routes without a feasible solution:", len(quick_travel_df) - len(quick_feasible_travel))
display(quick_travel_df.round(2))

In [ ]:
# Seed each elapsed-time run with the matching travel-time solution.
travel_result_by_route = {
    result["route_id"]: result
    for result in quick_travel_results
    if result["status"] == "feasible solution found"
}

quick_elapsed_results = []
for number, route_id in enumerate(travel_result_by_route, start=1):
    print(f"Elapsed objective {number}/{len(travel_result_by_route)}: {route_id}")
    result = optimize_one_route(
        route_id,
        time_limit_seconds=5,
        objective="elapsed",
    )
    quick_elapsed_results.append(result)
    print("Status:", result["status"])

quick_elapsed_df = pd.DataFrame([
    {key: value for key, value in result.items() if key != "optimized_stop_ids"}
    for result in quick_elapsed_results
])

comparison_columns = [
    "route_id",
    "optimized_travel_minutes",
    "optimized_elapsed_minutes",
    "optimized_wait_minutes",
]
quick_comparison_df = quick_travel_df[comparison_columns].merge(
    quick_elapsed_df[
        ["route_id", "status"] + comparison_columns[1:]
    ],
    on="route_id",
    suffixes=("_travel_objective", "_elapsed_objective"),
)
quick_comparison_df["elapsed_change_minutes"] = (
    quick_comparison_df["optimized_elapsed_minutes_elapsed_objective"]
    - quick_comparison_df["optimized_elapsed_minutes_travel_objective"]
)

print("Elapsed-time change: negative means the elapsed objective was faster.")
display(quick_comparison_df.round(2))

In [ ]:
retry_route_id = "RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c"

retry_elapsed_result = optimize_one_route(
    retry_route_id,
    time_limit_seconds=15,
    objective="elapsed",
)

for key, value in retry_elapsed_result.items():
    if key != "optimized_stop_ids":
        print(f"{key}: {value}")

In [ ]:
comparison_route_ids = [
    pilot_route_id,
    "RouteID_00573a3d-b909-42ee-8a99-c6474309d96b",
    "RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c",
    "RouteID_0091a35c-85fa-4b5e-90cf-4778f441753e",
    "RouteID_00a25e47-65c9-4efc-bfe9-8d9648a0df4e",
]

travel_comparison_results = []
elapsed_comparison_results = []

for number, route_id in enumerate(comparison_route_ids, start=1):
    # Use the same time limit for both objectives on each route.
    route_time_limit = 15 if route_id == (
        "RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c"
    ) else 5

    print(
        f"Comparing route {number}/{len(comparison_route_ids)}: "
        f"{route_id} (limit: {route_time_limit}s per objective)"
    )

    travel_result = optimize_one_route(
        route_id,
        time_limit_seconds=route_time_limit,
        objective="travel",
    )
    travel_comparison_results.append(travel_result)
    print("Travel objective:", travel_result["status"])

    elapsed_result = optimize_one_route(
        route_id,
        time_limit_seconds=route_time_limit,
        objective="elapsed",
    )
    elapsed_comparison_results.append(elapsed_result)
    print("Elapsed objective:", elapsed_result["status"])


comparison_columns = [
    "route_id",
    "status",
    "optimized_travel_minutes",
    "optimized_elapsed_minutes",
    "optimized_wait_minutes",
    "optimized_window_violations",
    "all_stops_visited_once",
]

travel_comparison_df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    }
    for result in travel_comparison_results
])[comparison_columns].rename(columns={
    "status": "travel_status",
    "optimized_travel_minutes": "travel_objective_drive_minutes",
    "optimized_elapsed_minutes": "travel_objective_elapsed_minutes",
    "optimized_wait_minutes": "travel_objective_wait_minutes",
    "optimized_window_violations": "travel_objective_window_violations",
    "all_stops_visited_once": "travel_objective_all_stops_visited",
})

elapsed_comparison_df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    }
    for result in elapsed_comparison_results
])[comparison_columns].rename(columns={
    "status": "elapsed_status",
    "optimized_travel_minutes": "elapsed_objective_drive_minutes",
    "optimized_elapsed_minutes": "elapsed_objective_elapsed_minutes",
    "optimized_wait_minutes": "elapsed_objective_wait_minutes",
    "optimized_window_violations": "elapsed_objective_window_violations",
    "all_stops_visited_once": "elapsed_objective_all_stops_visited",
})

objective_comparison_df = travel_comparison_df.merge(
    elapsed_comparison_df,
    on="route_id",
)

objective_comparison_df["elapsed_change_minutes"] = (
    objective_comparison_df["elapsed_objective_elapsed_minutes"]
    - objective_comparison_df["travel_objective_elapsed_minutes"]
)

display(objective_comparison_df.round(2))

In [ ]:
robustness_route_ids = [
    "RouteID_00573a3d-b909-42ee-8a99-c6474309d96b",
    "RouteID_0091a35c-85fa-4b5e-90cf-4778f441753e",
]

robustness_rows = []

for route_id in robustness_route_ids:
    for objective in ["travel", "elapsed"]:
        print(f"Solving {objective} objective for {route_id}")

        result = optimize_one_route(
            route_id,
            time_limit_seconds=15,
            objective=objective,
        )

        robustness_rows.append({
            "route_id": result["route_id"],
            "objective": result["objective"],
            "status": result["status"],
            "solver_status": result["solver_status"],
            "optimized_travel_minutes": result.get(
                "optimized_travel_minutes"
            ),
            "optimized_elapsed_minutes": result.get(
                "optimized_elapsed_minutes"
            ),
            "optimized_wait_minutes": result.get(
                "optimized_wait_minutes"
            ),
            "optimized_window_violations": result.get(
                "optimized_window_violations"
            ),
            "all_stops_visited_once": result.get(
                "all_stops_visited_once"
            ),
        })

robustness_df = pd.DataFrame(robustness_rows)

display(robustness_df.round(2))

In [ ]:
route_0074_long_search = optimize_one_route(
    "RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c",
    time_limit_seconds=30,
    objective="elapsed",
)

for key, value in route_0074_long_search.items():
    if key != "optimized_stop_ids":
        print(f"{key}: {value}")

In [ ]:
paired_route_ids = evaluation_route_ids[:10]
paired_results = []

for number, route_id in enumerate(paired_route_ids, start=1):
    for objective in ["travel", "elapsed"]:
        print(
            f"Route {number}/{len(paired_route_ids)} | "
            f"{objective}: {route_id}"
        )

        result = optimize_one_route(
            route_id,
            time_limit_seconds=5,
            objective=objective,
        )
        paired_results.append(result)
        print("Status:", result["status"])


paired_results_df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    }
    for result in paired_results
])

for objective in ["travel", "elapsed"]:
    objective_rows = paired_results_df[
        paired_results_df["objective"] == objective
    ]
    feasible_rows = objective_rows[
        objective_rows["status"] == "feasible solution found"
    ]

    print(f"\n{objective.title()} objective")
    print("Routes attempted:", len(objective_rows))
    print("Feasible solutions:", len(feasible_rows))
    print("Without a feasible solution:", len(objective_rows) - len(feasible_rows))

comparison_columns = [
    "route_id",
    "status",
    "optimized_travel_minutes",
    "optimized_elapsed_minutes",
    "optimized_wait_minutes",
    "optimized_window_violations",
    "all_stops_visited_once",
]

travel_results_df = paired_results_df[
    paired_results_df["objective"] == "travel"
][comparison_columns].rename(columns={
    "status": "travel_status",
    "optimized_travel_minutes": "travel_drive_minutes",
    "optimized_elapsed_minutes": "travel_elapsed_minutes",
    "optimized_wait_minutes": "travel_wait_minutes",
    "optimized_window_violations": "travel_window_violations",
    "all_stops_visited_once": "travel_all_stops_visited",
})

elapsed_results_df = paired_results_df[
    paired_results_df["objective"] == "elapsed"
][comparison_columns].rename(columns={
    "status": "elapsed_status",
    "optimized_travel_minutes": "elapsed_drive_minutes",
    "optimized_elapsed_minutes": "elapsed_route_minutes",
    "optimized_wait_minutes": "elapsed_wait_minutes",
    "optimized_window_violations": "elapsed_window_violations",
    "all_stops_visited_once": "elapsed_all_stops_visited",
})

paired_comparison_df = travel_results_df.merge(
    elapsed_results_df,
    on="route_id",
)

paired_comparison_df["elapsed_change_minutes"] = (
    paired_comparison_df["elapsed_route_minutes"]
    - paired_comparison_df["travel_elapsed_minutes"]
)

display(paired_comparison_df.round(2))

In [ ]:
route_1_retry_results = []

for objective in ["travel", "elapsed"]:
    print(f"Retrying route 1 with {objective} objective")

    result = optimize_one_route(
        "RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77",
        time_limit_seconds=15,
        objective=objective,
    )

    route_1_retry_results.append({
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    })

    print("Status:", result["status"])


route_1_retry_df = pd.DataFrame(route_1_retry_results)
display(route_1_retry_df.round(2))

In [28]:
from datetime import datetime, timezone
from ortools.constraint_solver import pywrapcp, routing_enums_pb2


def optimize_one_route(
    route_id,
    time_limit_seconds=15,
    objective="travel",
    initial_order=None,
):
    if time_limit_seconds <= 0:
        raise ValueError("time_limit_seconds must be greater than zero.")

    if objective not in {"travel", "elapsed"}:
        raise ValueError("objective must be 'travel' or 'elapsed'.")

    scale = 10
    route = route_json[route_id]
    packages_by_stop = evaluation_packages[route_id]
    travel_matrix = evaluation_matrices[route_id]

    sequence = actual_sequences[route_id]["actual"]
    ordered_stop_ids = [
        stop_id
        for stop_id, position in sorted(
            sequence.items(),
            key=lambda item: item[1],
        )
    ]

    expected_stop_ids = set(route["stops"])

    if not ordered_stop_ids or set(ordered_stop_ids) != expected_stop_ids:
        raise ValueError(f"Sequence does not match stops for {route_id}.")

    station_id = ordered_stop_ids[0]

    if route["stops"][station_id]["type"] != "Station":
        raise ValueError(f"Sequence does not start at station for {route_id}.")

    if set(packages_by_stop) != expected_stop_ids:
        raise ValueError(f"Package stops do not match for {route_id}.")

    if set(travel_matrix) != expected_stop_ids:
        raise ValueError(f"Travel-matrix origins do not match for {route_id}.")

    for origin in expected_stop_ids:
        if set(travel_matrix[origin]) != expected_stop_ids:
            raise ValueError(
                f"Travel-matrix destinations do not match at {origin}."
            )

    departure_time = datetime.fromisoformat(
        f"{route['date_YYYY_MM_DD']}T"
        f"{route['departure_time_utc']}+00:00"
    )

    def parse_utc(value):
        value = value.replace("Z", "+00:00")
        parsed = datetime.fromisoformat(value)

        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)

        return parsed.astimezone(timezone.utc)

    service_seconds_by_stop = {station_id: 0.0}
    window_seconds_by_stop = {}

    for stop_id in ordered_stop_ids[1:]:
        if route["stops"][stop_id]["type"] != "Dropoff":
            raise ValueError(f"Unexpected non-delivery stop: {stop_id}")

        service_times = []
        package_windows = []

        for details in packages_by_stop[stop_id].values():
            service_time = details.get("planned_service_time_seconds")

            if service_time is None:
                raise ValueError(f"Missing service time at stop {stop_id}.")

            service_times.append(float(service_time))

            window = details.get("time_window") or {}
            start = window.get("start_time_utc")
            end = window.get("end_time_utc")

            if isinstance(start, str) != isinstance(end, str):
                raise ValueError(f"Partial time window at stop {stop_id}.")

            if isinstance(start, str):
                start_time = parse_utc(start)
                end_time = parse_utc(end)

                if end_time < start_time:
                    raise ValueError(f"Invalid time window at stop {stop_id}.")

                package_windows.append((start_time, end_time))

        service_seconds_by_stop[stop_id] = sum(service_times)

        if package_windows:
            shared_start = max(start for start, end in package_windows)
            shared_end = min(end for start, end in package_windows)

            if shared_start > shared_end:
                raise ValueError(
                    f"No shared time window at stop {stop_id}."
                )

            window_seconds_by_stop[stop_id] = (
                (shared_start - departure_time).total_seconds(),
                (shared_end - departure_time).total_seconds(),
            )

    number_of_stops = len(ordered_stop_ids)
    dummy_end_node = number_of_stops

    travel_times_int = [
        [
            int(round(travel_matrix[origin][destination] * scale))
            for destination in ordered_stop_ids
        ]
        for origin in ordered_stop_ids
    ]

    service_times_int = [
        int(round(service_seconds_by_stop[stop_id] * scale))
        for stop_id in ordered_stop_ids
    ]

    time_windows_int = []

    for stop_id in ordered_stop_ids:
        if stop_id not in window_seconds_by_stop:
            time_windows_int.append(None)
            continue

        start_seconds, end_seconds = window_seconds_by_stop[stop_id]
        lower = int(round(max(0, start_seconds) * scale))
        upper = int(round(end_seconds * scale))

        if lower > upper:
            raise ValueError(
                f"Time window ends before departure at stop {stop_id}."
            )

        time_windows_int.append((lower, upper))

    # Dummy end: finish without returning to the station.
    for row in travel_times_int:
        row.append(0)

    travel_times_int.append([0] * (number_of_stops + 1))
    service_times_int.append(0)
    time_windows_int.append(None)

    max_window_end = max(
        (
            bounds[1]
            for bounds in time_windows_int[:number_of_stops]
            if bounds is not None
        ),
        default=0,
    )
    max_travel_time = max(max(row) for row in travel_times_int)
    total_service_time = sum(service_times_int)

    horizon = max(
        24 * 60 * 60 * scale,
        max_window_end
        + total_service_time
        + (number_of_stops + 1) * max_travel_time,
    )

    manager = pywrapcp.RoutingIndexManager(
        number_of_stops + 1,
        1,
        [0],
        [dummy_end_node],
    )
    routing = pywrapcp.RoutingModel(manager)

    def time_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        if to_node == dummy_end_node:
            return service_times_int[from_node]

        return (
            service_times_int[from_node]
            + travel_times_int[from_node][to_node]
        )

    time_callback_index = routing.RegisterTransitCallback(time_callback)
    routing.AddDimension(
        time_callback_index,
        horizon,
        horizon,
        True,
        "Time",
    )
    time_dimension = routing.GetDimensionOrDie("Time")

    def travel_cost_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        if to_node == dummy_end_node:
            return 0

        return travel_times_int[from_node][to_node]

    cost_index = routing.RegisterTransitCallback(travel_cost_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(cost_index)

    if objective == "elapsed":
        time_dimension.SetGlobalSpanCostCoefficient(horizon + 1)

    time_dimension.CumulVar(routing.Start(0)).SetRange(0, 0)

    for node in range(1, number_of_stops):
        index = manager.NodeToIndex(node)
        bounds = time_windows_int[node]

        if bounds is None:
            time_dimension.CumulVar(index).SetRange(0, horizon)
        else:
            time_dimension.CumulVar(index).SetRange(*bounds)

    time_dimension.CumulVar(routing.End(0)).SetRange(0, horizon)
    routing.AddVariableMinimizedByFinalizer(
        time_dimension.CumulVar(routing.End(0))
    )

    search = pywrapcp.DefaultRoutingSearchParameters()
    search.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    )
    search.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search.time_limit.seconds = int(time_limit_seconds)

    # Only call the potentially slow route-assignment loader when a seed
    # was explicitly supplied.
    initial_route_loaded = False

    if initial_order is not None:
        if (
            len(initial_order) != number_of_stops
            or set(initial_order) != expected_stop_ids
        ):
            raise ValueError(
                "Initial order must contain every route stop exactly once."
            )

        if initial_order[0] != station_id:
            raise ValueError("Initial order must start at the station.")

        node_by_stop_id = {
            stop_id: node
            for node, stop_id in enumerate(ordered_stop_ids)
        }
        route_nodes = [
            node_by_stop_id[stop_id]
            for stop_id in initial_order[1:]
        ]

        initial_assignment = routing.ReadAssignmentFromRoutes(
            [route_nodes],
            True,
        )

        if initial_assignment is not None:
            solution = routing.SolveFromAssignmentWithParameters(
                initial_assignment,
                search,
            )
            initial_route_loaded = True
        else:
            solution = routing.SolveWithParameters(search)
    else:
        solution = routing.SolveWithParameters(search)

    status_code = routing.status()
    status_names = {
        0: "not solved",
        1: "success",
        2: "partial success",
        3: "failure",
        4: "time limit reached before solution",
        5: "invalid model or parameters",
        6: "proven infeasible",
    }
    status = status_names.get(status_code, f"unknown status {status_code}")

    if solution is None:
        return {
            "route_id": route_id,
            "objective": objective,
            "status": status,
            "solver_status": status_code,
            "initial_route_loaded": initial_route_loaded,
        }

    optimized_stop_ids = []
    violations = []
    index = routing.Start(0)

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        stop_id = ordered_stop_ids[node]
        optimized_stop_ids.append(stop_id)

        bounds = time_windows_int[node]
        if bounds is not None:
            arrival = solution.Value(time_dimension.CumulVar(index))
            if not bounds[0] <= arrival <= bounds[1]:
                violations.append(stop_id)

        index = solution.Value(routing.NextVar(index))

    historical_travel = sum(
        travel_matrix[a][b]
        for a, b in zip(ordered_stop_ids, ordered_stop_ids[1:])
    )
    optimized_travel = sum(
        travel_matrix[a][b]
        for a, b in zip(optimized_stop_ids, optimized_stop_ids[1:])
    )

    current_time = 0.0
    historical_wait = 0.0
    historical_late = []

    for origin, destination in zip(
        ordered_stop_ids,
        ordered_stop_ids[1:],
    ):
        arrival = current_time + travel_matrix[origin][destination]
        service_start = arrival

        if destination in window_seconds_by_stop:
            start, end = window_seconds_by_stop[destination]
            service_start = max(arrival, start)
            historical_wait += service_start - arrival

            if service_start > end:
                historical_late.append(destination)

        current_time = (
            service_start + service_seconds_by_stop[destination]
        )

    historical_service = sum(service_seconds_by_stop.values())
    optimized_elapsed = (
        solution.Value(time_dimension.CumulVar(routing.End(0))) / scale
    )
    optimized_wait = max(
        0.0,
        optimized_elapsed - optimized_travel - historical_service,
    )

    return {
        "route_id": route_id,
        "objective": objective,
        "objective_definition": (
            "elapsed time primary, driving time tie-breaker"
            if objective == "elapsed"
            else "minimize driving time"
        ),
        "status": "feasible solution found",
        "solver_status": status_code,
        "initial_route_loaded": initial_route_loaded,
        "stop_count": number_of_stops,
        "historical_travel_minutes": historical_travel / 60,
        "optimized_travel_minutes": optimized_travel / 60,
        "travel_reduction_minutes": (historical_travel - optimized_travel) / 60,
        "travel_reduction_percent": (
            (historical_travel - optimized_travel)
            / historical_travel
            * 100
        ),
        "historical_elapsed_minutes": current_time / 60,
        "historical_wait_minutes": historical_wait / 60,
        "optimized_elapsed_minutes": optimized_elapsed / 60,
        "optimized_wait_minutes": optimized_wait / 60,
        "historical_late_stops": len(historical_late),
        "optimized_window_violations": len(violations),
        "all_stops_visited_once": (
            len(optimized_stop_ids) == number_of_stops
            and set(optimized_stop_ids) == expected_stop_ids
        ),
        "optimized_stop_ids": optimized_stop_ids,
    }

In [ ]:
route_1_order = [
    stop_id
    for stop_id, position in sorted(
        actual_sequences[
            "RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77"
        ]["actual"].items(),
        key=lambda item: item[1],
    )
]

route_1_seeded_results = []

for objective in ["travel", "elapsed"]:
    result = optimize_one_route(
        "RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77",
        time_limit_seconds=15,
        objective=objective,
        initial_order=route_1_order,
    )

    route_1_seeded_results.append({
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    })

    print(objective, "status:", result["status"])
    print("Initial route loaded:", result.get("initial_route_loaded"))

display(pd.DataFrame(route_1_seeded_results).round(2))

In [ ]:
paired_route_ids = evaluation_route_ids[:10]

route_1_id = "RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77"
route_6_id = "RouteID_0036cf37-1c93-48f8-b337-3dd6d1a195d8"

historical_route_1 = [
    stop_id
    for stop_id, position in sorted(
        actual_sequences[route_1_id]["actual"].items(),
        key=lambda item: item[1],
    )
]

historical_route_6 = [
    stop_id
    for stop_id, position in sorted(
        actual_sequences[route_6_id]["actual"].items(),
        key=lambda item: item[1],
    )
]

station_6 = historical_route_6[0]
late_stops_6 = ["CS", "GM"]
route_6_seed = (
    [station_6]
    + late_stops_6
    + [
        stop_id
        for stop_id in historical_route_6[1:]
        if stop_id not in late_stops_6
    ]
)

seed_orders = {
    route_1_id: historical_route_1,
    route_6_id: route_6_seed,
}

paired_results = []

for number, route_id in enumerate(paired_route_ids, start=1):
    route_seed = seed_orders.get(route_id)
    time_limit = 15 if route_seed is not None else 5

    for objective in ["travel", "elapsed"]:
        print(f"Route {number}/10 | {objective}: {route_id}")

        result = optimize_one_route(
            route_id,
            time_limit_seconds=time_limit,
            objective=objective,
            initial_order=route_seed,
        )
        paired_results.append(result)

        print(
            "Status:", result["status"],
            "| Starting route loaded:",
            result.get("initial_route_loaded"),
        )

paired_results_df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    }
    for result in paired_results
])

display(paired_results_df.round(2))

In [ ]:
remaining_route_ids = evaluation_route_ids[10:20]

for number, route_id in enumerate(remaining_route_ids, start=11):
    for objective in ["travel", "elapsed"]:
        print(f"Route {number}/20 | {objective}: {route_id}")

        result = optimize_one_route(
            route_id,
            time_limit_seconds=5,
            objective=objective,
            initial_order=None,
        )

        paired_results.append(result)

        print(
            "Status:", result["status"],
            "| Starting route loaded:",
            result.get("initial_route_loaded"),
        )


paired_results_df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key != "optimized_stop_ids"
    }
    for result in paired_results
])

print(
    "Total route-objective runs:",
    len(paired_results_df),
)
print(
    "Feasible runs:",
    int((paired_results_df["status"] == "feasible solution found").sum()),
)

display(
    paired_results_df[
        [
            "route_id",
            "objective",
            "status",
            "optimized_travel_minutes",
            "optimized_elapsed_minutes",
            "optimized_wait_minutes",
            "optimized_window_violations",
            "all_stops_visited_once",
        ]
    ].round(2)
)

In [ ]:
retry_route_ids = evaluation_route_ids[10:20]

routes_to_retry = [
    retry_route_ids[0],  # Route 11
    retry_route_ids[3],  # Route 14
    retry_route_ids[7],  # Route 18
]

retry_results = []

for route_id in routes_to_retry:
    sequence_info = actual_sequences[route_id]
    stop_positions = sequence_info["actual"]

    # Sort stop IDs by their historical sequence position.
    historical_order = [
        stop_id
        for stop_id, position in sorted(
            stop_positions.items(),
            key=lambda item: item[1],
        )
    ]

    for objective in ["travel", "elapsed"]:
        print(f"Retrying {route_id} | {objective} objective")

        result = optimize_one_route(
            route_id,
            time_limit_seconds=15,
            objective=objective,
            initial_order=historical_order,
        )

        retry_results.append(result)

        print(
            f"Status: {result.get('status')} | "
            f"Starting route loaded: {result.get('initial_route_loaded')}"
        )

retry_results_df = pd.DataFrame(retry_results)

display(
    retry_results_df[
        [
            "route_id",
            "objective",
            "status",
            "solver_status",
            "initial_route_loaded",
            "optimized_travel_minutes",
            "optimized_elapsed_minutes",
            "optimized_wait_minutes",
            "optimized_window_violations",
            "all_stops_visited_once",
        ]
    ]
)

In [ ]:
route_id = "RouteID_00595235-52e7-4b68-883c-70482701dea0"
sequence = actual_sequences[route_id]

print("Type:", type(sequence))
print("Length:", len(sequence) if hasattr(sequence, "__len__") else "no length")
print("Sample:", repr(sequence)[:1500])

In [ ]:
retry_results_df = pd.DataFrame(retry_results)

display(
    retry_results_df[
        [
            "route_id",
            "objective",
            "status",
            "solver_status",
            "initial_route_loaded",
            "optimized_travel_minutes",
            "optimized_elapsed_minutes",
            "optimized_wait_minutes",
            "optimized_window_violations",
            "all_stops_visited_once",
        ]
    ]
)

In [ ]:
from pathlib import Path
import pandas as pd

retry_route_ids = [
    "RouteID_00595235-52e7-4b68-883c-70482701dea0",  # Route 11
    "RouteID_0074e307-23c1-4b15-afa2-e58b45e4403c",  # Route 14
    "RouteID_0091a35c-85fa-4b5e-90cf-4778f441753e",  # Route 18
]

project_root = data_dir.parents[4]
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "route_retry_results.csv"

# Restore completed retries if this cell was run before.
if checkpoint_path.exists():
    retry_results_df = pd.read_csv(checkpoint_path)
else:
    retry_results_df = pd.DataFrame()

completed_pairs = set()
if not retry_results_df.empty:
    completed_pairs = set(
        zip(retry_results_df["route_id"], retry_results_df["objective"])
    )

for route_id in retry_route_ids:
    for objective in ["travel", "elapsed"]:
        if (route_id, objective) in completed_pairs:
            print(f"Already saved; skipping {route_id} | {objective}", flush=True)
            continue

        print(f"Solving {route_id} | {objective}", flush=True)

        try:
            # No initial_order: avoids ReadAssignmentFromRoutes.
            result = optimize_one_route(
                route_id,
                time_limit_seconds=20,
                objective=objective,
            )
            row = {
                key: value
                for key, value in result.items()
                if key != "optimized_stop_ids"
            }
        except Exception as error:
            row = {
                "route_id": route_id,
                "objective": objective,
                "status": "error",
                "error": repr(error),
            }

        retry_results_df = pd.concat(
            [retry_results_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        retry_results_df.to_csv(checkpoint_path, index=False)
        completed_pairs.add((route_id, objective))

        print(
            f"Saved: {row['route_id']} | {row['objective']} | "
            f"{row.get('status')}",
            flush=True,
        )

display(retry_results_df)
print("Checkpoint saved to:", checkpoint_path)

In [ ]:
objective_summary = []

for objective in ["travel", "elapsed"]:
    objective_df = feasible_df[
        feasible_df["objective"] == objective
    ].copy()

    objective_summary.append({
        "objective": objective,
        "feasible_runs": len(objective_df),
        "average_driving_minutes":
            objective_df["optimized_travel_minutes"].mean(),
        "median_driving_minutes":
            objective_df["optimized_travel_minutes"].median(),
        "average_elapsed_minutes":
            objective_df["optimized_elapsed_minutes"].mean(),
        "average_wait_minutes":
            objective_df["optimized_wait_minutes"].mean(),
        "routes_with_zero_window_violations": int(
            (objective_df["optimized_window_violations"] == 0).sum()
        ),
        "routes_visiting_every_stop_once": int(
            objective_df["all_stops_visited_once"].sum()
        ),
    })

objective_summary_df = pd.DataFrame(objective_summary)

travel_elapsed = feasible_df[
    feasible_df["objective"] == "travel"
][["route_id", "optimized_elapsed_minutes"]].rename(
    columns={
        "optimized_elapsed_minutes":
            "travel_objective_elapsed_minutes"
    }
)

elapsed_elapsed = feasible_df[
    feasible_df["objective"] == "elapsed"
][["route_id", "optimized_elapsed_minutes"]].rename(
    columns={
        "optimized_elapsed_minutes":
            "elapsed_objective_elapsed_minutes"
    }
)

paired_df = travel_elapsed.merge(elapsed_elapsed, on="route_id")

paired_df["elapsed_objective_change_minutes"] = (
    paired_df["elapsed_objective_elapsed_minutes"]
    - paired_df["travel_objective_elapsed_minutes"]
)

print("Saved route-objective runs:", len(results_df))
print("Feasible runs:", len(feasible_df))
print("Routes solved under both objectives:", len(paired_df))
print(
    "Routes where elapsed objective finished sooner:",
    int((paired_df["elapsed_objective_change_minutes"] < 0).sum()),
)
print(
    "Average elapsed-time change versus travel objective (minutes):",
    round(paired_df["elapsed_objective_change_minutes"].mean(), 2),
)

display(objective_summary_df.round(2))
display(paired_df.round(2))

In [ ]:
# Use only optimizer runs that found a feasible solution.
travel_runs = feasible_df[
    feasible_df["objective"] == "travel"
].copy()

elapsed_runs = feasible_df[
    feasible_df["objective"] == "elapsed"
].copy()

travel_runs["objective"] = "travel"
elapsed_runs["objective"] = "elapsed"

optimizer_runs = pd.concat(
    [travel_runs, elapsed_runs],
    ignore_index=True,
)

# Join each optimized route to its historical baseline.
baseline_comparison_df = optimizer_runs.merge(
    historical_audit_df,
    on="route_id",
    how="left",
    validate="many_to_one",
)

baseline_comparison_df["driving_reduction_percent"] = (
    (
        baseline_comparison_df["historical_travel_minutes"]
        - baseline_comparison_df["optimized_travel_minutes"]
    )
    / baseline_comparison_df["historical_travel_minutes"]
    * 100
)

baseline_comparison_df["elapsed_minutes_saved"] = (
    baseline_comparison_df["historical_elapsed_minutes"]
    - baseline_comparison_df["optimized_elapsed_minutes"]
)

summary_rows = []

for objective in ["travel", "elapsed"]:
    objective_df = baseline_comparison_df[
        baseline_comparison_df["objective"] == objective
    ]

    summary_rows.append({
        "objective": objective,
        "routes_with_feasible_solutions": len(objective_df),
        "average_driving_reduction_percent":
            objective_df["driving_reduction_percent"].mean(),
        "median_driving_reduction_percent":
            objective_df["driving_reduction_percent"].median(),
        "average_elapsed_minutes_saved":
            objective_df["elapsed_minutes_saved"].mean(),
        "routes_faster_than_historical": int(
            (objective_df["elapsed_minutes_saved"] > 0).sum()
        ),
        "routes_with_zero_window_violations": int(
            (objective_df["optimized_window_violations"] == 0).sum()
        ),
        "routes_visiting_every_stop_once": int(
            objective_df["all_stops_visited_once"].sum()
        ),
    })

baseline_summary_df = pd.DataFrame(summary_rows)

print(
    "Feasible route-objective comparisons with historical baselines:",
    len(baseline_comparison_df),
)

display(baseline_summary_df.round(2))

display(
    baseline_comparison_df[
        [
            "route_id",
            "objective",
            "historical_travel_minutes",
            "optimized_travel_minutes",
            "driving_reduction_percent",
            "historical_elapsed_minutes",
            "optimized_elapsed_minutes",
            "elapsed_minutes_saved",
            "historical_late_stops",
            "optimized_window_violations",
            "all_stops_visited_once",
        ]
    ].round(2)
)

In [ ]:
import numpy as np
import pandas as pd
import ijson

sample_size = 100
random_seed = 42
package_file = data_dir / "package_data.json"

# Count delivery stops that have at least one package with a complete time window.
windowed_stop_counts = {}

with package_file.open("rb") as raw_file:
    for route_id, packages_by_stop in ijson.kvitems(
        NaNReplacingReader(raw_file),
        "",
        use_float=True,
    ):
        if route_id not in route_json:
            continue

        route_stops = route_json[route_id]["stops"]
        windowed_count = 0

        for stop_id, packages in packages_by_stop.items():
            if (
                stop_id not in route_stops
                or route_stops[stop_id]["type"] != "Dropoff"
            ):
                continue

            stop_has_window = any(
                isinstance(
                    (package.get("time_window") or {}).get("start_time_utc"),
                    str,
                )
                and isinstance(
                    (package.get("time_window") or {}).get("end_time_utc"),
                    str,
                )
                for package in packages.values()
            )

            if stop_has_window:
                windowed_count += 1

        windowed_stop_counts[route_id] = windowed_count

# Make sure package data was found for every route.
missing_route_ids = set(route_json) - set(windowed_stop_counts)

if missing_route_ids:
    raise ValueError(
        f"Package data was not found for {len(missing_route_ids)} routes."
    )

# Build one feature row per route.
feature_rows = []

for route_id, route in route_json.items():
    stops = route["stops"]
    delivery_stop_count = sum(
        stop["type"] == "Dropoff"
        for stop in stops.values()
    )
    windowed_count = windowed_stop_counts[route_id]

    feature_rows.append({
        "route_id": route_id,
        "stop_count": len(stops),
        "delivery_stop_count": delivery_stop_count,
        "windowed_delivery_stop_count": windowed_count,
        "windowed_stop_rate": (
            windowed_count / delivery_stop_count
            if delivery_stop_count
            else 0.0
        ),
        "station_code": route.get("station_code"),
    })

route_features_df = pd.DataFrame(feature_rows)

if sample_size > len(route_features_df):
    raise ValueError(
        f"Requested {sample_size} routes, but only "
        f"{len(route_features_df)} routes are available."
    )

# Divide route size and time-window density into quartiles.
route_features_df["size_quartile"] = pd.qcut(
    route_features_df["stop_count"],
    q=4,
    labels=False,
    duplicates="drop",
)

route_features_df["window_density_quartile"] = pd.qcut(
    route_features_df["windowed_stop_rate"],
    q=4,
    labels=False,
    duplicates="drop",
)

# Allocate the 100 selections proportionally across the size/window groups.
strata_columns = ["size_quartile", "window_density_quartile"]
group_sizes = route_features_df.groupby(
    strata_columns,
    observed=True,
).size()

raw_allocations = group_sizes / len(route_features_df) * sample_size
allocations = np.floor(raw_allocations).astype(int)
remaining = sample_size - int(allocations.sum())

remainders = (raw_allocations - allocations).sort_values(ascending=False)

for stratum in remainders.index:
    if remaining == 0:
        break

    if allocations.loc[stratum] < group_sizes.loc[stratum]:
        allocations.loc[stratum] += 1
        remaining -= 1

if remaining:
    raise ValueError(
        f"Could not allocate all {sample_size} sample slots."
    )

# Sample reproducibly within each group.
selected_groups = []

for stratum, group in route_features_df.groupby(
    strata_columns,
    observed=True,
):
    number_to_select = int(allocations.loc[stratum])

    if number_to_select:
        selected_groups.append(
            group.sample(
                n=number_to_select,
                random_state=random_seed,
            )
        )

evaluation_routes_100_df = (
    pd.concat(selected_groups, ignore_index=True)
    .sort_values(
        ["size_quartile", "window_density_quartile", "route_id"]
    )
    .reset_index(drop=True)
)

if (
    len(evaluation_routes_100_df) != sample_size
    or not evaluation_routes_100_df["route_id"].is_unique
):
    raise ValueError("The selected sample is not 100 unique routes.")

# Save the sample so the same routes can be reused for later runs.
output_dir = data_dir.parents[4] / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

sample_path = output_dir / "evaluation_routes_100.csv"
evaluation_routes_100_df.to_csv(sample_path, index=False)

print("Routes in dataset:", len(route_features_df))
print("Routes selected:", len(evaluation_routes_100_df))
print("Random seed:", random_seed)
print("Saved sample to:", sample_path)

display(
    evaluation_routes_100_df[
        [
            "route_id",
            "station_code",
            "stop_count",
            "delivery_stop_count",
            "windowed_delivery_stop_count",
            "windowed_stop_rate",
            "size_quartile",
            "window_density_quartile",
        ]
    ].head(20)
)

print("Sample counts by route-size and window-density groups:")
display(
    evaluation_routes_100_df.groupby(
        strata_columns,
        observed=True,
    ).size().rename("sampled_routes").reset_index()
)

In [ ]:
import re

_nan_pattern = re.compile(rb"\bNaN\b")


def _is_word_byte(value):
    return (
        48 <= value <= 57
        or 65 <= value <= 90
        or 97 <= value <= 122
        or value == 95
    )


def _safe_json_cut(data):
    # Keep a possible incomplete NaN token until the next chunk arrives.
    for suffix in (b"NaN", b"Na", b"N"):
        if data.endswith(suffix):
            start = len(data) - len(suffix)

            if start == 0 or not _is_word_byte(data[start - 1]):
                return start

    return len(data)


class NaNReplacingReader:
    """Stream JSON bytes while safely replacing non-standard NaN values."""
    def __init__(self, raw_file, block_size=1_000_000):
        self.raw_file = raw_file
        self.block_size = block_size
        self.pending = b""
        self.ready = bytearray()
        self.eof = False

    def read(self, size=-1):
        if size == 0:
            return b""

        if size < 0:
            raise ValueError("The streaming parser requested an unbounded read.")

        while len(self.ready) < size and not self.eof:
            chunk = self.raw_file.read(
                max(self.block_size, size - len(self.ready))
            )

            if not chunk:
                self.ready.extend(_nan_pattern.sub(b"null", self.pending))
                self.pending = b""
                self.eof = True
                break

            combined = self.pending + chunk
            cut = _safe_json_cut(combined)

            self.ready.extend(
                _nan_pattern.sub(b"null", combined[:cut])
            )
            self.pending = combined[cut:]

        output = bytes(self.ready[:size])
        del self.ready[:size]
        return output


print("NaNReplacingReader updated.")

In [ ]:
selected_route_ids_100 = evaluation_routes_100_df["route_id"].tolist()
selected_route_set_100 = set(selected_route_ids_100)

packages_100 = {}
matrices_100 = {}

# Load package records for the 100 selected routes.
scanned = 0

with package_file.open("rb") as raw_file:
    for route_id, packages_by_stop in ijson.kvitems(
        NaNReplacingReader(raw_file),
        "",
        use_float=True,
    ):
        scanned += 1

        if route_id in selected_route_set_100:
            packages_100[route_id] = packages_by_stop

        if scanned % 500 == 0:
            print(
                f"Package file: scanned {scanned} routes; "
                f"selected {len(packages_100)}/100",
                flush=True,
            )

        if len(packages_100) == len(selected_route_set_100):
            break

if set(packages_100) != selected_route_set_100:
    missing = selected_route_set_100 - set(packages_100)
    raise ValueError(f"Missing package records for {len(missing)} selected routes.")

# Load travel matrices for those same 100 routes.
scanned = 0

with travel_file.open("rb") as raw_file:
    for route_id, travel_matrix in ijson.kvitems(
        NaNReplacingReader(raw_file),
        "",
        use_float=True,
    ):
        scanned += 1

        if route_id in selected_route_set_100:
            matrices_100[route_id] = travel_matrix

        if scanned % 500 == 0:
            print(
                f"Travel file: scanned {scanned} routes; "
                f"selected {len(matrices_100)}/100",
                flush=True,
            )

        if len(matrices_100) == len(selected_route_set_100):
            break

if set(matrices_100) != selected_route_set_100:
    missing = selected_route_set_100 - set(matrices_100)
    raise ValueError(f"Missing travel matrices for {len(missing)} selected routes.")

# Validate route stops, package stops, matrix stops, and historical sequences.
validation_rows = []

for route_id in selected_route_ids_100:
    expected_stops = set(route_json[route_id]["stops"])
    package_stops = set(packages_100[route_id])
    matrix_origins = set(matrices_100[route_id])
    sequence_stops = set(actual_sequences[route_id]["actual"])

    matrix_rows_match = all(
        set(row) == expected_stops
        for row in matrices_100[route_id].values()
    )

    validation_rows.append({
        "route_id": route_id,
        "stop_count": len(expected_stops),
        "package_stops_match": package_stops == expected_stops,
        "matrix_origins_match": matrix_origins == expected_stops,
        "matrix_rows_match": matrix_rows_match,
        "sequence_matches_stops": sequence_stops == expected_stops,
    })

validation_100_df = pd.DataFrame(validation_rows)

if not validation_100_df.drop(columns="route_id").all().all():
    display(validation_100_df[~validation_100_df.drop(
        columns="route_id"
    ).all(axis=1)])
    raise ValueError("At least one selected route failed the input checks.")

# Make the optimizer use the selected 100 routes.
evaluation_packages = packages_100
evaluation_matrices = matrices_100

print("Selected routes loaded:", len(selected_route_ids_100))
print("All four input checks passed for every selected route.")
display(validation_100_df.head(10))

In [ ]:
from pathlib import Path
import time
import pandas as pd

TIME_LIMIT_SECONDS = 5
OBJECTIVES = ["travel", "elapsed"]

sample_100_ids = evaluation_routes_100_df["route_id"].tolist()
sample_100_id_set = set(sample_100_ids)

checkpoint_path = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_results.csv"
)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

# Load any previous checkpoints and make sure they belong to this same sample.
if checkpoint_path.exists():
    benchmark_results_df = pd.read_csv(checkpoint_path)

    saved_route_ids = set(benchmark_results_df["route_id"].astype(str))
    unexpected_route_ids = saved_route_ids - sample_100_id_set

    if unexpected_route_ids:
        raise ValueError(
            "The checkpoint contains routes outside the current 100-route sample. "
            "Stop here and tell me before replacing it."
        )
else:
    benchmark_results_df = pd.DataFrame()

completed_pairs = set()

if not benchmark_results_df.empty:
    completed_pairs = {
        (str(row["route_id"]), str(row["objective"]))
        for _, row in benchmark_results_df.iterrows()
        if row.get("status") != "error"
    }

for route_number, route_id in enumerate(sample_100_ids, start=1):
    for objective in OBJECTIVES:
        pair = (route_id, objective)

        if pair in completed_pairs:
            print(
                f"Already saved; skipping {route_number}/100 "
                f"| {objective} | {route_id}",
                flush=True,
            )
            continue

        print(
            f"Route {route_number}/100 | {objective} | {route_id}",
            flush=True,
        )

        start_time = time.perf_counter()

        try:
            # No initial_order: this avoids the slow historical-route loader.
            result = optimize_one_route(
                route_id,
                time_limit_seconds=TIME_LIMIT_SECONDS,
                objective=objective,
            )

            row = {
                key: value
                for key, value in result.items()
                if key != "optimized_stop_ids"
            }
            row["sample_number"] = route_number
            row["time_limit_seconds"] = TIME_LIMIT_SECONDS
            row["run_seconds"] = round(
                time.perf_counter() - start_time,
                2,
            )

        except Exception as error:
            row = {
                "route_id": route_id,
                "objective": objective,
                "status": "error",
                "error": repr(error),
                "sample_number": route_number,
                "time_limit_seconds": TIME_LIMIT_SECONDS,
                "run_seconds": round(
                    time.perf_counter() - start_time,
                    2,
                ),
            }

        benchmark_results_df = pd.concat(
            [benchmark_results_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        benchmark_results_df.to_csv(checkpoint_path, index=False)

        completed_pairs.add(pair)

        print(
            f"Saved | status: {row.get('status')} "
            f"| run time: {row['run_seconds']} sec",
            flush=True,
        )

# Summarize completed runs and paired route coverage.
feasible_results_df = benchmark_results_df[
    benchmark_results_df["status"] == "feasible solution found"
].copy()

print("\nRoute-objective runs saved:", len(benchmark_results_df))
print("Feasible runs:", len(feasible_results_df))
print("Checkpoint:", checkpoint_path)

display(
    benchmark_results_df.groupby(
        ["objective", "status"],
        dropna=False,
    ).size().rename("runs").reset_index()
)

status_by_route_df = benchmark_results_df.pivot_table(
    index="route_id",
    columns="objective",
    values="status",
    aggfunc="first",
)

for objective in OBJECTIVES:
    if objective not in status_by_route_df.columns:
        status_by_route_df[objective] = None

both_objectives_feasible = (
    (status_by_route_df["travel"] == "feasible solution found")
    & (status_by_route_df["elapsed"] == "feasible solution found")
)

print(
    "Routes with feasible solutions for both objectives:",
    int(both_objectives_feasible.sum()),
)

display(
    feasible_results_df[
        [
            "route_id",
            "objective",
            "status",
            "optimized_travel_minutes",
            "optimized_elapsed_minutes",
            "optimized_wait_minutes",
            "optimized_window_violations",
            "all_stops_visited_once",
            "run_seconds",
        ]
    ].round(2)
)

In [ ]:
import pandas as pd

checkpoint_path = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_results.csv"
)

benchmark_results_df = pd.read_csv(checkpoint_path)

# Treat saved results as feasible only when the optimizer returned a solution.
benchmark_results_df["is_feasible"] = (
    benchmark_results_df["status"] == "feasible solution found"
)

benchmark_results_df["elapsed_saved_vs_historical_minutes"] = (
    pd.to_numeric(
        benchmark_results_df["historical_elapsed_minutes"],
        errors="coerce",
    )
    - pd.to_numeric(
        benchmark_results_df["optimized_elapsed_minutes"],
        errors="coerce",
    )
)

feasible_results_df = benchmark_results_df[
    benchmark_results_df["is_feasible"]
].copy()

# Objective-level performance, compared with the historical route.
objective_summary_df = (
    feasible_results_df
    .groupby("objective")
    .agg(
        feasible_runs=("route_id", "size"),
        average_driving_reduction_percent=(
            "travel_reduction_percent", "mean"
        ),
        median_driving_reduction_percent=(
            "travel_reduction_percent", "median"
        ),
        average_elapsed_minutes_saved=(
            "elapsed_saved_vs_historical_minutes", "mean"
        ),
        median_elapsed_minutes_saved=(
            "elapsed_saved_vs_historical_minutes", "median"
        ),
        routes_faster_than_historical=(
            "elapsed_saved_vs_historical_minutes",
            lambda values: int((values > 0).sum()),
        ),
        average_wait_minutes=("optimized_wait_minutes", "mean"),
        window_violations=("optimized_window_violations", "sum"),
        routes_visiting_every_stop_once=(
            "all_stops_visited_once", "sum"
        ),
        average_solver_seconds=("run_seconds", "mean"),
    )
    .reset_index()
)

objective_summary_df["solve_rate_percent"] = (
    objective_summary_df["feasible_runs"] / 100 * 100
)

# Compare objectives only for routes where both returned feasible solutions.
travel_runs = feasible_results_df[
    feasible_results_df["objective"] == "travel"
][
    [
        "route_id",
        "optimized_travel_minutes",
        "optimized_elapsed_minutes",
        "optimized_wait_minutes",
    ]
].rename(columns={
    "optimized_travel_minutes": "travel_drive_minutes",
    "optimized_elapsed_minutes": "travel_elapsed_minutes",
    "optimized_wait_minutes": "travel_wait_minutes",
})

elapsed_runs = feasible_results_df[
    feasible_results_df["objective"] == "elapsed"
][
    [
        "route_id",
        "optimized_travel_minutes",
        "optimized_elapsed_minutes",
        "optimized_wait_minutes",
    ]
].rename(columns={
    "optimized_travel_minutes": "elapsed_objective_drive_minutes",
    "optimized_elapsed_minutes": "elapsed_objective_elapsed_minutes",
    "optimized_wait_minutes": "elapsed_objective_wait_minutes",
})

benchmark_paired_df = travel_runs.merge(
    elapsed_runs,
    on="route_id",
    how="inner",
    validate="one_to_one",
)

benchmark_paired_df["elapsed_objective_drive_change_minutes"] = (
    benchmark_paired_df["elapsed_objective_drive_minutes"]
    - benchmark_paired_df["travel_drive_minutes"]
)

benchmark_paired_df["elapsed_objective_elapsed_change_minutes"] = (
    benchmark_paired_df["elapsed_objective_elapsed_minutes"]
    - benchmark_paired_df["travel_elapsed_minutes"]
)

print("Route-objective runs:", len(benchmark_results_df))
print("Feasible route-objective runs:", len(feasible_results_df))
print("Routes solved under both objectives:", len(benchmark_paired_df))

display(objective_summary_df.round(2))

print(
    "Routes where elapsed objective finished sooner than travel objective:",
    int(
        (
            benchmark_paired_df[
                "elapsed_objective_elapsed_change_minutes"
            ] < 0
        ).sum()
    ),
)
print(
    "Mean elapsed-time change versus travel objective (minutes):",
    round(
        benchmark_paired_df[
            "elapsed_objective_elapsed_change_minutes"
        ].mean(),
        2,
    ),
)
print(
    "Median elapsed-time change versus travel objective (minutes):",
    round(
        benchmark_paired_df[
            "elapsed_objective_elapsed_change_minutes"
        ].median(),
        2,
    ),
)

# Check whether solve rates vary by route size or time-window density.
sample_features = evaluation_routes_100_df[
    [
        "route_id",
        "size_quartile",
        "window_density_quartile",
    ]
]

strata_results_df = benchmark_results_df.merge(
    sample_features,
    on="route_id",
    how="left",
    validate="many_to_one",
)

strata_summary_df = (
    strata_results_df
    .groupby(
        ["objective", "size_quartile", "window_density_quartile"],
        observed=True,
    )
    .agg(
        attempted_runs=("route_id", "size"),
        feasible_runs=("is_feasible", "sum"),
    )
    .reset_index()
)

strata_summary_df["solve_rate_percent"] = (
    strata_summary_df["feasible_runs"]
    / strata_summary_df["attempted_runs"]
    * 100
)

display(strata_summary_df.round(1))

# Save analysis tables for the project report.
output_dir = checkpoint_path.parent
objective_summary_df.to_csv(
    output_dir / "optimization_100_objective_summary.csv",
    index=False,
)
benchmark_paired_df.to_csv(
    output_dir / "optimization_100_paired_comparison.csv",
    index=False,
)
strata_summary_df.to_csv(
    output_dir / "optimization_100_strata_solve_rates.csv",
    index=False,
)

print("Summary tables saved in:", output_dir)

In [ ]:
import time
import pandas as pd

original_results_path = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_results.csv"
)

original_results = pd.read_csv(original_results_path)

status_by_route = original_results.pivot_table(
    index="route_id",
    columns="objective",
    values="status",
    aggfunc="first",
)

unresolved_ids = status_by_route.index[
    (status_by_route.get("travel") != "feasible solution found")
    | (status_by_route.get("elapsed") != "feasible solution found")
].tolist()

unresolved_features = evaluation_routes_100_df[
    evaluation_routes_100_df["route_id"].isin(unresolved_ids)
].copy()

retry_sample_df = unresolved_features.sample(
    n=min(10, len(unresolved_features)),
    random_state=42,
).reset_index(drop=True)

retry_route_ids = retry_sample_df["route_id"].tolist()

print("Unresolved routes:", len(unresolved_ids))
print("Routes selected for the 15-second test:", len(retry_route_ids))
display(
    retry_sample_df[
        [
            "route_id",
            "stop_count",
            "windowed_delivery_stop_count",
            "windowed_stop_rate",
            "size_quartile",
            "window_density_quartile",
        ]
    ]
)

retry_checkpoint_path = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_timeout_sensitivity_15s.csv"
)

if retry_checkpoint_path.exists():
    retry_results_df = pd.read_csv(retry_checkpoint_path)

    unexpected_ids = (
        set(retry_results_df["route_id"].astype(str))
        - set(retry_route_ids)
    )
    if unexpected_ids:
        raise ValueError(
            "The 15-second checkpoint contains routes outside this retry sample."
        )
else:
    retry_results_df = pd.DataFrame()

completed_retry_pairs = set()

if not retry_results_df.empty:
    completed_retry_pairs = {
        (str(row["route_id"]), str(row["objective"]))
        for _, row in retry_results_df.iterrows()
        if row.get("status") != "error"
    }

for number, route_id in enumerate(retry_route_ids, start=1):
    for objective in ["travel", "elapsed"]:
        pair = (route_id, objective)

        if pair in completed_retry_pairs:
            print(f"Already saved; skipping {route_id} | {objective}", flush=True)
            continue

        print(
            f"15-second retry {number}/{len(retry_route_ids)} "
            f"| {objective} | {route_id}",
            flush=True,
        )

        started = time.perf_counter()

        try:
            # No historical starting order: avoids the route-loader hang.
            result = optimize_one_route(
                route_id,
                time_limit_seconds=15,
                objective=objective,
            )

            row = {
                key: value
                for key, value in result.items()
                if key != "optimized_stop_ids"
            }
            row["run_seconds"] = round(time.perf_counter() - started, 2)

        except Exception as error:
            row = {
                "route_id": route_id,
                "objective": objective,
                "status": "error",
                "error": repr(error),
                "run_seconds": round(time.perf_counter() - started, 2),
            }

        retry_results_df = pd.concat(
            [retry_results_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        retry_results_df.to_csv(retry_checkpoint_path, index=False)
        completed_retry_pairs.add(pair)

        print(
            f"Saved | {row.get('status')} "
            f"| {row['run_seconds']} sec",
            flush=True,
        )

print("15-second retry results saved to:", retry_checkpoint_path)
display(retry_results_df)

In [ ]:
from datetime import datetime, timezone
import pandas as pd


def parse_audit_utc(value):
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    return parsed.astimezone(timezone.utc)


def audit_historical_route(route_id):
    route = route_json[route_id]
    packages_by_stop = evaluation_packages[route_id]
    travel_matrix = evaluation_matrices[route_id]

    sequence = actual_sequences[route_id]["actual"]
    ordered_stop_ids = [
        stop_id
        for stop_id, position in sorted(
            sequence.items(),
            key=lambda item: item[1],
        )
    ]

    expected_stops = set(route["stops"])

    if set(ordered_stop_ids) != expected_stops:
        return {
            "route_id": route_id,
            "historical_window_feasible": False,
            "late_stop_count": None,
            "conflicting_window_stop_count": None,
            "reason": "Historical sequence and route stops differ",
        }

    departure_time = datetime.fromisoformat(
        f"{route['date_YYYY_MM_DD']}T"
        f"{route['departure_time_utc']}+00:00"
    )

    service_seconds = {}
    windows_seconds = {}
    conflicting_window_stops = []

    for stop_id in ordered_stop_ids:
        if route["stops"][stop_id]["type"] != "Dropoff":
            service_seconds[stop_id] = 0.0
            continue

        package_windows = []
        stop_service_seconds = 0.0

        for package in packages_by_stop[stop_id].values():
            planned_service = package.get("planned_service_time_seconds")

            if planned_service is None:
                raise ValueError(
                    f"Missing service time for {route_id}, stop {stop_id}"
                )

            stop_service_seconds += float(planned_service)

            time_window = package.get("time_window") or {}
            start = time_window.get("start_time_utc")
            end = time_window.get("end_time_utc")

            if isinstance(start, str) and isinstance(end, str):
                package_windows.append((
                    parse_audit_utc(start),
                    parse_audit_utc(end),
                ))

        service_seconds[stop_id] = stop_service_seconds

        if package_windows:
            shared_start = max(start for start, end in package_windows)
            shared_end = min(end for start, end in package_windows)

            if shared_start > shared_end:
                conflicting_window_stops.append(stop_id)
            else:
                windows_seconds[stop_id] = (
                    (shared_start - departure_time).total_seconds(),
                    (shared_end - departure_time).total_seconds(),
                )

    current_time = 0.0
    waiting_seconds = 0.0
    late_stops = []

    for origin, destination in zip(
        ordered_stop_ids,
        ordered_stop_ids[1:],
    ):
        arrival = current_time + travel_matrix[origin][destination]
        service_start = arrival

        if destination in windows_seconds:
            window_start, window_end = windows_seconds[destination]
            service_start = max(arrival, window_start)
            waiting_seconds += service_start - arrival

            if service_start > window_end:
                late_stops.append(destination)

        current_time = service_start + service_seconds[destination]

    return {
        "route_id": route_id,
        "historical_window_feasible": (
            len(late_stops) == 0
            and len(conflicting_window_stops) == 0
        ),
        "late_stop_count": len(late_stops),
        "conflicting_window_stop_count": len(conflicting_window_stops),
        "historical_travel_minutes": sum(
            travel_matrix[a][b]
            for a, b in zip(ordered_stop_ids, ordered_stop_ids[1:])
        ) / 60,
        "historical_wait_minutes": waiting_seconds / 60,
        "historical_elapsed_minutes": current_time / 60,
        "reason": "",
    }


unresolved_audit_df = pd.DataFrame([
    audit_historical_route(route_id)
    for route_id in unresolved_ids
])

print("Unresolved routes audited:", len(unresolved_audit_df))
print(
    "Historical routes feasible under our hard time windows:",
    int(unresolved_audit_df["historical_window_feasible"].sum()),
)
print(
    "Historical routes with late or conflicting windows:",
    int((~unresolved_audit_df["historical_window_feasible"]).sum()),
)

display(
    unresolved_audit_df.sort_values(
        ["historical_window_feasible", "late_stop_count"],
        ascending=[True, False],
    )
)

print("\nThe 10 routes tested at 15 seconds:")
display(
    unresolved_audit_df[
        unresolved_audit_df["route_id"].isin(retry_route_ids)
    ].sort_values("route_id")
)

In [29]:
import time
import pandas as pd

known_feasible_retry_ids = unresolved_audit_df.loc[
    unresolved_audit_df["route_id"].isin(retry_route_ids)
    & unresolved_audit_df["historical_window_feasible"],
    "route_id",
].tolist()

print(
    "Routes in this test with historically feasible orders:",
    len(known_feasible_retry_ids),
)

if not known_feasible_retry_ids:
    raise ValueError("No historically feasible routes were selected.")

insertion_checkpoint_path = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_insertion_strategy_15s.csv"
)

if insertion_checkpoint_path.exists():
    insertion_results_df = pd.read_csv(insertion_checkpoint_path)

    unexpected_ids = (
        set(insertion_results_df["route_id"].astype(str))
        - set(known_feasible_retry_ids)
    )
    if unexpected_ids:
        raise ValueError(
            "This checkpoint contains routes outside the current test."
        )
else:
    insertion_results_df = pd.DataFrame()

completed_pairs = set()

if not insertion_results_df.empty:
    completed_pairs = {
        (str(row["route_id"]), str(row["objective"]))
        for _, row in insertion_results_df.iterrows()
    }

for number, route_id in enumerate(known_feasible_retry_ids, start=1):
    for objective in ["travel", "elapsed"]:
        if (route_id, objective) in completed_pairs:
            print(f"Already saved; skipping {route_id} | {objective}", flush=True)
            continue

        print(
            f"Insertion strategy {number}/{len(known_feasible_retry_ids)} "
            f"| {objective} | {route_id}",
            flush=True,
        )

        started = time.perf_counter()

        try:
            result = optimize_one_route(
                route_id,
                time_limit_seconds=15,
                objective=objective,
            )

            row = {
                key: value
                for key, value in result.items()
                if key != "optimized_stop_ids"
            }
            row["run_seconds"] = round(time.perf_counter() - started, 2)

        except Exception as error:
            row = {
                "route_id": route_id,
                "objective": objective,
                "status": "error",
                "error": repr(error),
                "run_seconds": round(time.perf_counter() - started, 2),
            }

        insertion_results_df = pd.concat(
            [insertion_results_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        insertion_results_df.to_csv(insertion_checkpoint_path, index=False)
        completed_pairs.add((route_id, objective))

        print(
            f"Saved | {row.get('status')} | {row['run_seconds']} sec",
            flush=True,
        )

print("Saved insertion-strategy test to:", insertion_checkpoint_path)
display(insertion_results_df)

Routes in this test with historically feasible orders: 8
Insertion strategy 1/8 | travel | RouteID_1a02d09f-2690-4564-87fc-b6d36045e5ba
Saved | feasible solution found | 15.0 sec
Insertion strategy 1/8 | elapsed | RouteID_1a02d09f-2690-4564-87fc-b6d36045e5ba
Saved | feasible solution found | 15.0 sec
Insertion strategy 2/8 | travel | RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4
Saved | failure | 0.22 sec
Insertion strategy 2/8 | elapsed | RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4
Saved | failure | 0.23 sec
Insertion strategy 3/8 | travel | RouteID_67188405-b57c-4d2b-aa0a-40d860c34e72
Saved | failure | 0.32 sec
Insertion strategy 3/8 | elapsed | RouteID_67188405-b57c-4d2b-aa0a-40d860c34e72
Saved | failure | 0.32 sec
Insertion strategy 4/8 | travel | RouteID_85e93874-76f4-43c1-82c8-cf7866500638
Saved | feasible solution found | 15.0 sec
Insertion strategy 4/8 | elapsed | RouteID_85e93874-76f4-43c1-82c8-cf7866500638
Saved | feasible solution found | 15.0 sec
Insertion strategy 5/8 | tr

,route_id,objective,objective_definition,status,solver_status,initial_route_loaded,stop_count,historical_travel_minutes,optimized_travel_minutes,travel_reduction_minutes,travel_reduction_percent,historical_elapsed_minutes,historical_wait_minutes,optimized_elapsed_minutes,optimized_wait_minutes,historical_late_stops,optimized_window_violations,all_stops_visited_once,run_seconds
0,RouteID_1a02d09f-2690-4564-87fc-b6d36045e5ba,travel,minimize driving time,feasible solution found,1,False,70.0,84.336667,73.073333,11.263333,13.355203,456.188333,10.868333,446.893333,1.283667e+01,0.0,0.0,True,15.00
1,RouteID_1a02d09f-2690-4564-87fc-b6d36045e5ba,elapsed,"elapsed time primary, driving time tie-breaker",feasible solution found,1,False,70.0,84.336667,78.243333,6.093333,7.225011,456.188333,10.868333,439.226667,0.000000e+00,0.0,0.0,True,15.00
2,RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4,travel,NaN,failure,3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.22
3,RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4,elapsed,NaN,failure,3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.23
4,RouteID_67188405-b57c-4d2b-aa0a-40d860c34e72,travel,NaN,failure,3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.32
5,RouteID_67188405-b57c-4d2b-aa0a-40d860c34e72,elapsed,NaN,failure,3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.32
6,RouteID_85e93874-76f4-43c1-82c8-cf7866500638,travel,minimize driving time,feasible solution found,1,False,138.0,204.330000,171.283333,33.046667,16.173184,495.743333,25.923333,462.696667,2.592333e+01,0.0,0.0,True,15.00
7,RouteID_85e93874-76f4-43c1-82c8-cf7866500638,elapsed,"elapsed time primary, driving time tie-breaker",feasible solution found,1,False,138.0,204.330000,177.660000,26.670000,13.052415,495.743333,25.923333,443.150000,0.000000e+00,0.0,0.0,True,15.00
8,RouteID_9cfa02c6-2d81-42ec-9448-b7696f54cf39,travel,NaN,failure,3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.25
9,RouteID_9cfa02c6-2d81-42ec-9448-b7696f54cf39,elapsed,NaN,failure,3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.25


In [30]:
import time
import pandas as pd

insertion_candidate_ids = unresolved_audit_df.loc[
    unresolved_audit_df["historical_window_feasible"],
    "route_id",
].tolist()

print(
    "Unresolved routes with historically feasible orders:",
    len(insertion_candidate_ids),
)

insertion_checkpoint_path = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_insertion_strategy_15s.csv"
)

if insertion_checkpoint_path.exists():
    insertion_results_df = pd.read_csv(insertion_checkpoint_path)

    unexpected_ids = (
        set(insertion_results_df["route_id"].astype(str))
        - set(insertion_candidate_ids)
    )
    if unexpected_ids:
        raise ValueError(
            "The insertion checkpoint includes routes outside this group."
        )
else:
    insertion_results_df = pd.DataFrame()

completed_pairs = set()

if not insertion_results_df.empty:
    completed_pairs = {
        (str(row["route_id"]), str(row["objective"]))
        for _, row in insertion_results_df.iterrows()
        if row.get("status") != "error"
    }

for number, route_id in enumerate(insertion_candidate_ids, start=1):
    for objective in ["travel", "elapsed"]:
        if (route_id, objective) in completed_pairs:
            print(
                f"Already saved; skipping {number}/{len(insertion_candidate_ids)} "
                f"| {objective} | {route_id}",
                flush=True,
            )
            continue

        print(
            f"Insertion test {number}/{len(insertion_candidate_ids)} "
            f"| {objective} | {route_id}",
            flush=True,
        )

        started = time.perf_counter()

        try:
            # Cell 21 must currently use PARALLEL_CHEAPEST_INSERTION.
            # No initial_order is supplied.
            result = optimize_one_route(
                route_id,
                time_limit_seconds=15,
                objective=objective,
            )

            row = {
                key: value
                for key, value in result.items()
                if key != "optimized_stop_ids"
            }
            row["run_seconds"] = round(time.perf_counter() - started, 2)

        except Exception as error:
            row = {
                "route_id": route_id,
                "objective": objective,
                "status": "error",
                "error": repr(error),
                "run_seconds": round(time.perf_counter() - started, 2),
            }

        insertion_results_df = pd.concat(
            [insertion_results_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        insertion_results_df.to_csv(insertion_checkpoint_path, index=False)
        completed_pairs.add((route_id, objective))

        print(
            f"Saved | {row.get('status')} | {row['run_seconds']} sec",
            flush=True,
        )

print("Insertion-strategy checkpoint:", insertion_checkpoint_path)
display(
    insertion_results_df.groupby(
        ["objective", "status"],
        dropna=False,
    ).size().rename("runs").reset_index()
)

Unresolved routes with historically feasible orders: 28
Already saved; skipping 1/28 | travel | RouteID_1a02d09f-2690-4564-87fc-b6d36045e5ba
Already saved; skipping 1/28 | elapsed | RouteID_1a02d09f-2690-4564-87fc-b6d36045e5ba
Already saved; skipping 2/28 | travel | RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4
Already saved; skipping 2/28 | elapsed | RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4
Insertion test 3/28 | travel | RouteID_2838e580-84d0-414b-b931-acbc9159af6f
Saved | feasible solution found | 15.0 sec
Insertion test 3/28 | elapsed | RouteID_2838e580-84d0-414b-b931-acbc9159af6f
Saved | feasible solution found | 15.0 sec
Insertion test 4/28 | travel | RouteID_32a8b599-90ab-4f07-af83-9ae9c4bc65d4
Saved | feasible solution found | 15.0 sec
Insertion test 4/28 | elapsed | RouteID_32a8b599-90ab-4f07-af83-9ae9c4bc65d4
Saved | feasible solution found | 15.0 sec
Insertion test 5/28 | travel | RouteID_3a426557-3c49-42c6-a10e-c453606fcad5
Saved | feasible solution found | 15.0 sec
Inser

,objective,status,runs
0,elapsed,failure,5
1,elapsed,feasible solution found,23
2,travel,failure,5
3,travel,feasible solution found,23


In [31]:
import pandas as pd

output_dir = data_dir.parents[4] / "data" / "processed"

baseline_df = pd.read_csv(
    output_dir / "optimization_100_results.csv"
)
insertion_df = pd.read_csv(
    output_dir / "optimization_100_insertion_strategy_15s.csv"
)

# Summarize the insertion test against historical route performance.
insertion_feasible_df = insertion_df[
    insertion_df["status"] == "feasible solution found"
].copy()

insertion_feasible_df["elapsed_saved_vs_historical_minutes"] = (
    insertion_feasible_df["historical_elapsed_minutes"]
    - insertion_feasible_df["optimized_elapsed_minutes"]
)

insertion_summary_df = (
    insertion_feasible_df
    .groupby("objective")
    .agg(
        feasible_runs=("route_id", "size"),
        average_driving_reduction_percent=(
            "travel_reduction_percent", "mean"
        ),
        median_driving_reduction_percent=(
            "travel_reduction_percent", "median"
        ),
        average_elapsed_minutes_saved=(
            "elapsed_saved_vs_historical_minutes", "mean"
        ),
        routes_with_zero_window_violations=(
            "optimized_window_violations",
            lambda values: int((values == 0).sum()),
        ),
        routes_visiting_every_stop_once=(
            "all_stops_visited_once", "sum"
        ),
    )
    .reset_index()
)

insertion_status_pivot = insertion_df.pivot_table(
    index="route_id",
    columns="objective",
    values="status",
    aggfunc="first",
)

for objective in ["travel", "elapsed"]:
    if objective not in insertion_status_pivot.columns:
        insertion_status_pivot[objective] = None

insertion_both_feasible = (
    (insertion_status_pivot["travel"] == "feasible solution found")
    & (insertion_status_pivot["elapsed"] == "feasible solution found")
)

print("Routes in insertion test:", insertion_df["route_id"].nunique())
print(
    "Insertion-test routes feasible for both objectives:",
    int(insertion_both_feasible.sum()),
)
display(insertion_summary_df.round(2))

# Count route/objective pairs solved by either configuration.
baseline_success = baseline_df[
    baseline_df["status"] == "feasible solution found"
]
insertion_success = insertion_df[
    insertion_df["status"] == "feasible solution found"
]

solved_pairs = set(
    zip(baseline_success["route_id"], baseline_success["objective"])
)
solved_pairs.update(
    zip(insertion_success["route_id"], insertion_success["objective"])
)

sample_route_ids = evaluation_routes_100_df["route_id"].tolist()

routes_solved_for_both = [
    route_id
    for route_id in sample_route_ids
    if (route_id, "travel") in solved_pairs
    and (route_id, "elapsed") in solved_pairs
]

print(
    "\nRoutes with a feasible result for both objectives "
    "in at least one configuration:",
    len(routes_solved_for_both),
)
print(
    "Routes still missing a feasible result for at least one objective:",
    100 - len(routes_solved_for_both),
)

Routes in insertion test: 28
Insertion-test routes feasible for both objectives: 23


,objective,feasible_runs,average_driving_reduction_percent,median_driving_reduction_percent,average_elapsed_minutes_saved,routes_with_zero_window_violations,routes_visiting_every_stop_once
0,elapsed,23,9.59,7.68,23.09,23,23
1,travel,23,12.03,12.16,1.24,23,23



Routes with a feasible result for both objectives in at least one configuration: 86
Routes still missing a feasible result for at least one objective: 14


In [32]:
import time
import pandas as pd

historically_infeasible_ids = unresolved_audit_df.loc[
    ~unresolved_audit_df["historical_window_feasible"],
    "route_id",
].tolist()

print(
    "Routes with late or conflicting historical windows:",
    len(historically_infeasible_ids),
)

late_route_checkpoint = (
    data_dir.parents[4]
    / "data"
    / "processed"
    / "optimization_100_historical_late_insertion_15s.csv"
)

if late_route_checkpoint.exists():
    late_retry_results_df = pd.read_csv(late_route_checkpoint)

    unexpected_ids = (
        set(late_retry_results_df["route_id"].astype(str))
        - set(historically_infeasible_ids)
    )
    if unexpected_ids:
        raise ValueError(
            "This checkpoint contains routes outside the selected group."
        )
else:
    late_retry_results_df = pd.DataFrame()

completed_pairs = set()

if not late_retry_results_df.empty:
    completed_pairs = {
        (str(row["route_id"]), str(row["objective"]))
        for _, row in late_retry_results_df.iterrows()
        if row.get("status") != "error"
    }

for number, route_id in enumerate(
    historically_infeasible_ids,
    start=1,
):
    for objective in ["travel", "elapsed"]:
        if (route_id, objective) in completed_pairs:
            print(f"Already saved; skipping {route_id} | {objective}", flush=True)
            continue

        print(
            f"Late-history test {number}/{len(historically_infeasible_ids)} "
            f"| {objective} | {route_id}",
            flush=True,
        )

        started = time.perf_counter()

        try:
            result = optimize_one_route(
                route_id,
                time_limit_seconds=15,
                objective=objective,
            )

            row = {
                key: value
                for key, value in result.items()
                if key != "optimized_stop_ids"
            }
            row["run_seconds"] = round(time.perf_counter() - started, 2)

        except Exception as error:
            row = {
                "route_id": route_id,
                "objective": objective,
                "status": "error",
                "error": repr(error),
                "run_seconds": round(time.perf_counter() - started, 2),
            }

        late_retry_results_df = pd.concat(
            [late_retry_results_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        late_retry_results_df.to_csv(late_route_checkpoint, index=False)
        completed_pairs.add((route_id, objective))

        print(
            f"Saved | {row.get('status')} | {row['run_seconds']} sec",
            flush=True,
        )

print("Results saved to:", late_route_checkpoint)
display(
    late_retry_results_df.groupby(
        ["objective", "status"],
        dropna=False,
    ).size().rename("runs").reset_index()
)

Routes with late or conflicting historical windows: 9
Late-history test 1/9 | travel | RouteID_410cfd1a-a562-4d20-95a0-e0e084573f25
Saved | feasible solution found | 15.01 sec
Late-history test 1/9 | elapsed | RouteID_410cfd1a-a562-4d20-95a0-e0e084573f25
Saved | feasible solution found | 15.0 sec
Late-history test 2/9 | travel | RouteID_461362c5-9c9a-4570-b98f-f8617a4e3de4
Saved | feasible solution found | 15.0 sec
Late-history test 2/9 | elapsed | RouteID_461362c5-9c9a-4570-b98f-f8617a4e3de4
Saved | feasible solution found | 15.0 sec
Late-history test 3/9 | travel | RouteID_5f8d9c67-df02-453d-93d2-429a2c21a2c1
Saved | feasible solution found | 15.0 sec
Late-history test 3/9 | elapsed | RouteID_5f8d9c67-df02-453d-93d2-429a2c21a2c1
Saved | feasible solution found | 15.0 sec
Late-history test 4/9 | travel | RouteID_6b4fde35-3e5d-4545-b5b5-f8e76d07f13b
Saved | failure | 0.1 sec
Late-history test 4/9 | elapsed | RouteID_6b4fde35-3e5d-4545-b5b5-f8e76d07f13b
Saved | failure | 0.11 sec
Late-h

,objective,status,runs
0,elapsed,failure,2
1,elapsed,feasible solution found,7
2,travel,failure,2
3,travel,feasible solution found,7


In [34]:
import pandas as pd

output_dir = data_dir.parents[4] / "data" / "processed"

baseline_df = pd.read_csv(
    output_dir / "optimization_100_results.csv"
)
feasible_retry_df = pd.read_csv(
    output_dir / "optimization_100_insertion_strategy_15s.csv"
)
late_retry_df = pd.read_csv(
    output_dir / "optimization_100_historical_late_insertion_15s.csv"
)

# Keep only feasible records from each stage and label their solver settings.
baseline_success_df = baseline_df[
    baseline_df["status"] == "feasible solution found"
].copy()
baseline_success_df["solver_configuration"] = (
    "PATH_CHEAPEST_ARC, 5 seconds"
)

feasible_retry_success_df = feasible_retry_df[
    feasible_retry_df["status"] == "feasible solution found"
].copy()
feasible_retry_success_df["solver_configuration"] = (
    "PARALLEL_CHEAPEST_INSERTION, 15 seconds"
)

late_retry_success_df = late_retry_df[
    late_retry_df["status"] == "feasible solution found"
].copy()
late_retry_success_df["solver_configuration"] = (
    "PARALLEL_CHEAPEST_INSERTION, 15 seconds"
)

all_successful_results_df = pd.concat(
    [
        baseline_success_df,
        feasible_retry_success_df,
        late_retry_success_df,
    ],
    ignore_index=True,
)

# A route/objective pair should contribute only one successful result.
duplicate_pairs = all_successful_results_df.duplicated(
    ["route_id", "objective"],
    keep=False,
)

if duplicate_pairs.any():
    raise ValueError(
        "More than one successful result exists for a route/objective pair. "
        "Review those rows before summarizing."
    )

all_successful_results_df["elapsed_saved_vs_historical_minutes"] = (
    all_successful_results_df["historical_elapsed_minutes"]
    - all_successful_results_df["optimized_elapsed_minutes"]
)

# Validate feasibility checks across all successful results.
all_successful_results_df["all_stops_visited_once"] = (
    all_successful_results_df["all_stops_visited_once"]
    .astype(str)
    .str.strip()
    .map({"True": True, "False": False})
)

print("Successful route-objective results:", len(all_successful_results_df))
print(
    "Successful results with zero time-window violations:",
    int(
        (
            all_successful_results_df["optimized_window_violations"] == 0
        ).sum()
    ),
)
print(
    "Successful results visiting every stop once:",
    int(all_successful_results_df["all_stops_visited_once"].sum()),
)

# Summarize each objective against its historical route.
summary_rows = []

for objective in ["travel", "elapsed"]:
    objective_df = all_successful_results_df[
        all_successful_results_df["objective"] == objective
    ]

    summary_rows.append({
        "objective": objective,
        "feasible_routes": objective_df["route_id"].nunique(),
        "solve_rate_percent": (
            objective_df["route_id"].nunique() / 100 * 100
        ),
        "average_driving_reduction_percent":
            objective_df["travel_reduction_percent"].mean(),
        "median_driving_reduction_percent":
            objective_df["travel_reduction_percent"].median(),
        "average_elapsed_minutes_saved":
            objective_df["elapsed_saved_vs_historical_minutes"].mean(),
        "median_elapsed_minutes_saved":
            objective_df["elapsed_saved_vs_historical_minutes"].median(),
        "routes_faster_than_historical": int(
            (
                objective_df["elapsed_saved_vs_historical_minutes"] > 0
            ).sum()
        ),
        "average_wait_minutes":
            objective_df["optimized_wait_minutes"].mean(),
    })

final_summary_df = pd.DataFrame(summary_rows)

# Compare objectives only on routes with a successful result for both.
travel_df = all_successful_results_df[
    all_successful_results_df["objective"] == "travel"
][
    ["route_id", "optimized_travel_minutes", "optimized_elapsed_minutes"]
].rename(columns={
    "optimized_travel_minutes": "travel_objective_drive_minutes",
    "optimized_elapsed_minutes": "travel_objective_elapsed_minutes",
})

elapsed_df = all_successful_results_df[
    all_successful_results_df["objective"] == "elapsed"
][
    ["route_id", "optimized_travel_minutes", "optimized_elapsed_minutes"]
].rename(columns={
    "optimized_travel_minutes": "elapsed_objective_drive_minutes",
    "optimized_elapsed_minutes": "elapsed_objective_elapsed_minutes",
})

final_paired_df = travel_df.merge(
    elapsed_df,
    on="route_id",
    how="inner",
    validate="one_to_one",
)

final_paired_df["elapsed_objective_elapsed_change_minutes"] = (
    final_paired_df["elapsed_objective_elapsed_minutes"]
    - final_paired_df["travel_objective_elapsed_minutes"]
)

print(
    "\nRoutes with feasible results for both objectives:",
    len(final_paired_df),
)
print(
    "Routes where elapsed objective finished sooner:",
    int(
        (
            final_paired_df["elapsed_objective_elapsed_change_minutes"] < 0
        ).sum()
    ),
)
print(
    "Mean elapsed-time change versus travel objective (minutes):",
    round(
        final_paired_df[
            "elapsed_objective_elapsed_change_minutes"
        ].mean(),
        2,
    ),
)
print(
    "Median elapsed-time change versus travel objective (minutes):",
    round(
        final_paired_df[
            "elapsed_objective_elapsed_change_minutes"
        ].median(),
        2,
    ),
)

# List routes that still lack a feasible result for at least one objective.
solved_route_ids = set(final_paired_df["route_id"])
unresolved_final_df = evaluation_routes_100_df[
    ~evaluation_routes_100_df["route_id"].isin(solved_route_ids)
].merge(
    unresolved_audit_df[
        [
            "route_id",
            "historical_window_feasible",
            "late_stop_count",
            "conflicting_window_stop_count",
        ]
    ],
    on="route_id",
    how="left",
)

display(final_summary_df.round(2))
display(unresolved_final_df)

# Save the combined report tables.
all_successful_results_df.to_csv(
    output_dir / "optimization_100_successful_results.csv",
    index=False,
)
final_summary_df.to_csv(
    output_dir / "optimization_100_final_summary.csv",
    index=False,
)
final_paired_df.to_csv(
    output_dir / "optimization_100_final_paired_comparison.csv",
    index=False,
)
unresolved_final_df.to_csv(
    output_dir / "optimization_100_unresolved_routes.csv",
    index=False,
)

print("Final report tables saved in:", output_dir)

Successful route-objective results: 186
Successful results with zero time-window violations: 186
Successful results visiting every stop once: 186

Routes with feasible results for both objectives: 93
Routes where elapsed objective finished sooner: 31
Mean elapsed-time change versus travel objective (minutes): -13.91
Median elapsed-time change versus travel objective (minutes): 0.45


,objective,feasible_routes,solve_rate_percent,average_driving_reduction_percent,median_driving_reduction_percent,average_elapsed_minutes_saved,median_elapsed_minutes_saved,routes_faster_than_historical,average_wait_minutes
0,travel,93,93.0,12.52,12.16,7.0,17.56,73,17.80
1,elapsed,93,93.0,10.28,9.12,20.9,18.56,93,0.26


,route_id,stop_count,delivery_stop_count,windowed_delivery_stop_count,windowed_stop_rate,station_code,size_quartile,window_density_quartile,historical_window_feasible,late_stop_count,conflicting_window_stop_count
0,RouteID_f328bffe-bf90-4ddd-b80a-a3fd78695a6f,117,116,24,0.206897,DLA4,0,3,True,0,0
1,RouteID_6b4fde35-3e5d-4545-b5b5-f8e76d07f13b,144,143,24,0.167832,DSE4,1,3,False,1,0
2,RouteID_d31a808e-1cab-4330-9461-6b38866dbb9f,153,152,17,0.111842,DLA7,2,3,False,2,0
3,RouteID_67188405-b57c-4d2b-aa0a-40d860c34e72,200,199,10,0.050251,DCH3,3,1,True,0,0
4,RouteID_26e8b143-7eb9-4636-8fb9-7698e28038d4,180,179,12,0.067039,DCH2,3,2,True,0,0
5,RouteID_faaf69b7-d9da-4147-a56d-abac94a7bfba,195,194,12,0.061856,DCH3,3,2,True,0,0
6,RouteID_9cfa02c6-2d81-42ec-9448-b7696f54cf39,186,185,32,0.172973,DCH2,3,3,True,0,0


Final report tables saved in: /Users/harshitbhargava/Desktop/LMD Project/data/processed
